# The Geography of Immigration Enforcement: Exploring Border Surveillance Patterns and IIRIRA 287(g) County Participation in Texas
Shalini Das<br>
GGIS 407
## 1. Introduction
Section 287(g) agreements under the IIRIRA (Illegal Immigration Reform and Immigrant Responsibility Act) of 1996. is also of heightened interest for immigrants, attornerys, civil rights advocates, and enforcement agencies alike.  between 1994 and January 2026, where more than 150 out of 254 counties prior to July 2025 had already entered into 

I will evaluate the relationship between ICE detention centers, immigration court locations, and surveillance tower networks to capture the full arc from the emergence of "Prevention Through Deterrence" in the 1990s through the Obama era "Secure Communities" (S-Comm) program up to present-day data sharing partnerships between Immigrations and Customs Enforcement (ICE), commercial data brokers, government databases, and local law enforcement. My focus is on the State of Texas  between 1994 and January 2026. Due to a State Senate Bill (SB 8) passed in July 2025 as part of Governor Greg Abbott's 'Operation Lone Star' program that started in 2021, soon all eligible Texas Counties will be required to enter into a 287(g) agreement, i.e. 234 of 254 counties, by the end of December 2026. This will effectively force local and state authorities to collaborate with ICE/CBP and grant local law enforcement to assume federal immigration enforcement powers through three types of programs: Warrant Service Office (WSO) Model, Jail Enforcement Model (JEM), or Task Force Model (TFM).

In [1]:
import os
import geopandas as gpd 
from geopandas import GeoSeries, GeoDataFrame
import contextily as cx
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from IPython.display import HTML, display
from typing import Dict, List, Tuple
import pandas as pd
import numpy as np
from shapely.geometry import Point
from sqlalchemy import create_engine
import osmnx as ox
import networkx as nx
import folium
from folium import GeoJsonTooltip
import json

ox.settings.use_cache = True
ox.settings.log_console = True

## 2. Data Ingestion and Preprocessing

### Tower Data

In [2]:
towers = pd.read_csv('/home/jovyan/work/GGIS 407/Final Project/data/EFF_towers_01_26.csv')
towers = towers[towers['State of Tower'] == 'Texas']
towers = towers[towers['Sector'] != 'Tucson Sector']
towers = towers.iloc[:,0:23]

# Convert to GeoDataFrame (this makes sure it spatial)
geometry = [Point(xy) for xy in zip(towers['Longitude'], towers['Latitude'])] 
towers_gdf = gpd.GeoDataFrame(towers, geometry=geometry, crs='EPSG:4326')
towers_gdf = towers_gdf[['Tower Name', 'Current Tower Type', 'Tower Subcategory', 'Sector', 'Sector Acronym', 'Station/AOR', 'Vendor', 'geometry']]
towers_gdf.head()

,Tower Name,Current Tower Type,Tower Subcategory,Sector,Sector Acronym,Station/AOR,Vendor,geometry
0,BRP Customs B&M,Remote Video Surveillance System (RVSS),Relocatable Tower,Rio Grande Valley Sector,RGV,Brownsville Station,General Dynamics,POINT (-97.50435 25.89390)
1,BRP Mulberry,Remote Video Surveillance System (RVSS),Relocatable Tower,Rio Grande Valley Sector,RGV,Brownsville Station,General Dynamics,POINT (-97.59932 25.96807)
4,DRT-DRS-02 Upper Weir2,Remote Video Surveillance System (RVSS),Monopole Tower,Del Rio Sector,DRT,Del Rio Station,General Dynamics,POINT (-101.04079 29.42519)
5,DRT-DRS-06 Lower Weir Dam2,Remote Video Surveillance System (RVSS),Monopole Tower,Del Rio Sector,DRT,Del Rio Station,NaN,POINT (-100.93011 29.32899)
6,DRT-DRS-07 POE Bridge,Remote Video Surveillance System (RVSS),Monopole Tower,Del Rio Sector,DRT,Del Rio Station,NaN,POINT (-100.92446 29.32991)


In [3]:
towers_gdf.dtypes

Tower Name              object
Current Tower Type      object
Tower Subcategory       object
Sector                  object
Sector Acronym          object
Station/AOR             object
Vendor                  object
geometry              geometry
dtype: object

In [4]:
towers_gdf.to_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/towers_gdf.geojson", driver='GeoJSON')

In [5]:
cols_to_convert = ['Current Tower Type', 'Tower Subcategory', 'Sector', 'Sector Acronym', 'Station/AOR', 'Vendor']
towers_gdf[cols_to_convert] = towers_gdf[cols_to_convert].astype('category')
towers_gdf.dtypes

for col in cols_to_convert:
    print(f"{towers_gdf[col].nunique()} {col} Types")
    print(f"{towers_gdf[col].value_counts()}\n")

3 Current Tower Type Types
Remote Video Surveillance System (RVSS)    173
Autonomous Surveillance Tower (AST)        109
Uncategorized                               11
Name: Current Tower Type, dtype: int64

3 Tower Subcategory Types
Monopole Tower                 80
Relocatable Tower              60
Extended Range Sentry Tower     2
Name: Tower Subcategory, dtype: int64

5 Sector Types
Rio Grande Valley Sector    92
Laredo Sector               69
Big Bend Sector             49
Del Rio Sector              44
El Paso Sector              28
Name: Sector, dtype: int64

6 Sector Acronym Types
RGV    90
LRT    69
BBT    49
DRT    44
EPT    28
RGC     2
Name: Sector Acronym, dtype: int64

15 Station/AOR Types
El Paso Station            20
Harlingen Station          18
Brownsville Station        12
Rio Grande City Station    12
McAllen station            10
Weslaco Station            10
Del Rio Station             9
Fort Brown Station          8
Laredo West Station         5
Laredo South Stat

### Detention Centers (Vera Institute/ and TRAC) Data

In [6]:
facilities_10_25 = pd.read_csv('/home/jovyan/work/GGIS 407/Final Project/data/facilities.csv')
facilities_10_25 = facilities_10_25[facilities_10_25['state'] == 'TX']
geometry = [Point(xy) for xy in zip(facilities_10_25['longitude'], facilities_10_25['latitude'])] 
facilities_gdf = gpd.GeoDataFrame(facilities_10_25, geometry=geometry, crs='EPSG:4326')
facilities_gdf.head()
display(HTML(f'<div style="height: 300px; overflow: auto; width: fit-content">{facilities_gdf.to_html()}</div>'))

,detention_facility_code,detention_facility_name,address,city,county,state,zip,aor,latitude,longitude,type_detailed,type_grouped,geometry
3,ABRMCTX,Abilene Regional Med Ctr,6250 Us-83,Abilene,Taylor,TX,79606.0,Dallas,32.359857,-99.743409,Hospital,Medical,POINT (-99.74341 32.35986)
5,ABTHOLD,Abilene Hold Room,12071 Fm 3522,Abilene,Jones,TX,79601.0,Dallas,32.562128,-99.635383,Hold,Hold/Staging,POINT (-99.63538 32.56213)
36,ALAMOTX,Alamo Police Dept.,840 W Austin St,Alamo,Hidalgo,TX,78516.0,Harlingen,26.181502,-98.119115,Unknown,Other/Unknown,POINT (-98.11911 26.18150)
48,AMTHOLD,Amarillo Hold Room,8601 E. Amarillo Blvd,Amarillo,Potter,TX,79108.0,Dallas,35.222536,-101.748848,Hold,Hold/Staging,POINT (-101.74885 35.22254)
52,ANDRWTX,Andrews Sherrif's Office Co Jail,201 N. Main,Andrews,Andrews,TX,79714.0,El Paso,32.320133,-102.547641,Other,Other/Unknown,POINT (-102.54764 32.32013)
54,ANSGHTX,Anson General Hospital,101 Avenue J,Anson,Jones,TX,79501.0,Dallas,32.767880,-99.895188,Hospital,Medical,POINT (-99.89519 32.76788)
67,ASWHRTX,Ascension Seton Williams Hosp,201 Seton Parkway,Round Rock,Williamson,TX,78665.0,San Antonio,30.567070,-97.651251,Hospital,Medical,POINT (-97.65125 30.56707)
70,ATTHOLD,"Athens, TX Hold Room",109 W Corsicana Street,Athens,Henderson,TX,75751.0,Dallas,32.204448,-95.853981,Hold,Hold/Staging,POINT (-95.85398 32.20445)
72,AUSHOLD,Austin DRO Hold Room,300 East 8th Street,Austin,Travis,TX,78701.0,San Antonio,30.269638,-97.738978,Hold,Hold/Staging,POINT (-97.73898 30.26964)
93,BEDCITX,Bedford City Jail,2121 L Don Dodson Drive,Bedford,Tarrant,TX,76021.0,Dallas,32.841090,-97.133092,IGSA,Non-Dedicated,POINT (-97.13309 32.84109)


In [7]:
detention_centers_gdf = facilities_gdf[facilities_gdf['type_detailed'] != 'Hospital']
display(HTML(f'<div style="height: 300px; overflow: auto; width: fit-content">{detention_centers_gdf.to_html()}</div>'))

,detention_facility_code,detention_facility_name,address,city,county,state,zip,aor,latitude,longitude,type_detailed,type_grouped,geometry
5,ABTHOLD,Abilene Hold Room,12071 Fm 3522,Abilene,Jones,TX,79601.0,Dallas,32.562128,-99.635383,Hold,Hold/Staging,POINT (-99.63538 32.56213)
36,ALAMOTX,Alamo Police Dept.,840 W Austin St,Alamo,Hidalgo,TX,78516.0,Harlingen,26.181502,-98.119115,Unknown,Other/Unknown,POINT (-98.11911 26.18150)
48,AMTHOLD,Amarillo Hold Room,8601 E. Amarillo Blvd,Amarillo,Potter,TX,79108.0,Dallas,35.222536,-101.748848,Hold,Hold/Staging,POINT (-101.74885 35.22254)
52,ANDRWTX,Andrews Sherrif's Office Co Jail,201 N. Main,Andrews,Andrews,TX,79714.0,El Paso,32.320133,-102.547641,Other,Other/Unknown,POINT (-102.54764 32.32013)
70,ATTHOLD,"Athens, TX Hold Room",109 W Corsicana Street,Athens,Henderson,TX,75751.0,Dallas,32.204448,-95.853981,Hold,Hold/Staging,POINT (-95.85398 32.20445)
72,AUSHOLD,Austin DRO Hold Room,300 East 8th Street,Austin,Travis,TX,78701.0,San Antonio,30.269638,-97.738978,Hold,Hold/Staging,POINT (-97.73898 30.26964)
93,BEDCITX,Bedford City Jail,2121 L Don Dodson Drive,Bedford,Tarrant,TX,76021.0,Dallas,32.841090,-97.133092,IGSA,Non-Dedicated,POINT (-97.13309 32.84109)
99,BELCOTX,Bell County Jail,111 W. Central,Belton,Bell,TX,76513.0,Houston,31.057053,-97.463733,IGSA,Non-Dedicated,POINT (-97.46373 31.05705)
106,BFGHITX,"Byrd's Foster Group Home, Inc",5708 Hardy St,Houston,Harris,TX,77009.0,Houston,29.811312,-95.352550,Juvenile,Family/Youth,POINT (-95.35255 29.81131)
118,BLBNATX,Bluebonnet Det Fclty,400 2nd Street,Anson,Jones,TX,79501.0,Dallas,32.778038,-99.909668,IGSA,Non-Dedicated,POINT (-99.90967 32.77804)


In [8]:
# detention_centers_gdf = detention_centers_gdf.set_crs('EPSG:4326', allow_override=True)
detention_centers_gdf.to_file("detention_centers_gdf.geojson", driver='GeoJSON')

### Immigration Courts Data (Scraping)
Executive Office of Immigration Review (EOIR) of the DOJ (Department of Justice)

In [9]:
def get_texas_judges() -> Dict[str, List[str]]:
    """
    Create dictionary with list of all Judge Names that correspond to each Texas Immigration Court (dictionary key). 
    Source: https://www.justice.gov/eoir/find-immigration-court-and-access-internet-based-hearings#TX
    """ 
    
    texas_judges = {
        "Conroe Immigration Court": [
            "ACIJ Daniel P. Kinnicutt",
            "Chris Brisack",
            "Andrew Caborn",
            "Andrea Cole",
            "Timothy M. Cole",
            "Holly D'Andrea",
            "Mark Evans",
            "John McPhail",
            "Robert Powell",
            "Billy Sapp"],
        
        "Dallas Immigration Court": [
            "ACIJ Tara Naselow-Nahas",
            "Attila Bogdan",
            "Xiomara Davis-Gumbs",
            "Charissa Dvorak",
            "Cait Irwin",
            "Kyle Mitchell",
            "James Nugent",
            "Richard Ozmun",
            "Deitrich Sims",
            "Christopher Thielemann",
            "Jennifer Winfield"],
        
        "El Paso Immigration Court": [
            "ACIJ Nathan Herbert",
            "Janette Allen",
            "Judith Bonilla",
            "Lorely Fernandez",
            "Daniel J. Gallegos",
            "Steven Meredith",
            "James Miller Jr.",
            "Danny Razo"],
        
        "El Paso SPC Immigration Court": [
            "ACIJ Nathan Herbert",
            "Michael Pleters",
            "Jessica Miles",
            "Stephen Ruhle",
            "Dean Tuckman"],
        
        "Fort Worth IAC Immigration Court": [
            "ACIJ Tara Naselow-Nahas",
            "Carol Bell",
            "Robert Boudreau",
            "D'Anna H. Freeman",
            "Donald Eller",
            "Jacob Bashore",
            "Charles Dishman",
            "Randall Fluke",
            "David Green",
            "Cynthia Goodman",
            "Allan John-Baptiste",
            "Brandon Hart",
            "Richard Jacobs",
            "Hugo Martinez",
            "Jennifer May",
            "Francis Mwangi",
            "James F. Polivka",
            "Kenneth Shahan"],
        
        "Harlingen Immigration Court": [
            "ACIJ Melissa Joy Garcia",
            "Julian Castaneda",
            "Delia I Gonzalez",
            "Joseph Leonard",
            "Christopher Pineda"],
        
        "Houston - Jefferson Street Immigration Court": [
            "ACIJ Daniel P. Kinnicutt",
            "Nimmo Bhagat",
            "Sam Brown",
            "Kevin N. Buras",
            "Gary Endelman",
            "Scott V. Greenbaum",
            "Saul Greenstein",
            "Anwer Khan",
            "Nicholas B. Lucic",
            "Bao Nguyen",
            "Maritza Ramos",
            "Abdias Tida",
            "Lynn Wang"],
        
        "Houston - Greenspoint Park Immigration Court": [
            "ACIJ Adam Jovanovic",
            "Justin Dinsdale",
            "Danielle Garten",
            "Monica Guidry",
            "Nicholle Hempel",
            "Maria Jaimes-Salgado",
            "Brent Landis",
            "Alexander Lee",
            "Joshua Osborn",
            "Martinque Parker",
            "Alex Perez",
            "Christopher Phan",
            "Artie Pobjecky",
            "Oshea Spencer",
            "Jason L. Stern",
            "Lydia Tamez",
            "Kenya Wells"],
        
        "Houston - S. Gessner Road Immigration Court": [
            "ACIJ Adam Jovanovic",
            "Jonathan Andresen",
            "Kevin Brown",
            "Miguel Cordero-Gonzalez",
            "Geoffrey Hoffman",
            "Arkesia Jenkins-Dargan",
            "DeLana Jones",
            "Thomas Johnson",
            "Jacquelyn Jo. Joyce",
            "David M. Paxton",
            "Lionel Martin",
            "Jared Robinson",
            "Christopher Schumann",
            "Andrew X. Stawar"],
        
        "Laredo Immigration Court": [
            "ACIJ Melissa Joy Garcia",
            "Roel Canales",
            "Laura Figueroa",
            "Emmanuel Garcia",
            "Courtney Stevens",
            "Jesus J. Ybarra"],
        
        "Pearsall Immigration Court": [
            "ACIJ Jayme Salinardi",
            "Stuart Alcorn",
            "Linda L. Chavez",
            "Robert R. McKee",
            "Jennifer L. Mazza",
            "Veronica Segovia",
            "Kevin Terrill"],

        "Port Isabel Immigration Court": [
            "ACIJ Melissa Joy Garcia",
            "Paul Hable",
            "Frank Pimentel",
            "Margaret R. MacGregor"],
        
        "San Antonio Immigration Court": [
            "ACIJ Jayme Salinardi",
            "Justin Adams",
            "Margaret Burkhart",
            "Thomas Crossan",
            "Yvonne Gonzalez",
            "Cynthia LaFuente",
            "Anibal Martinez",
            "Elizabeth Martinez",
            "Charles McCullough",
            "Rifian Newaz",
            "Daniel Santander",
            "Eric Tijerina",
            "Meredith Tyrakoski"]
    }
    
    return texas_judges

In [10]:
def get_texas_facilities() -> List[Tuple[str, str, str, str]]:
    """ 
    Create lists of all Detention Facilities, Service Processing Centers, and Other Hearing Locations under the Assigned Responsibility of each Texas Administrative Control Immigration Court, organizing results into: ['facility_name', 'facility_type', 'court_name', 'court_address']. 
    Source: https://www.justice.gov/eoir/immigration-court-administrative-control-list
    """
    
    facilities = [
        # Conroe Immigration Court
        ("Montgomery Processing Center (CIC)", "Processing Center", "Conroe Immigration Court", 
         "806 Hilbig Road, Conroe, TX 77301"),
        ("Houston Service Processing Center", "Service Processing Center", "Conroe Immigration Court", 
         "5520 Greens Road, Houston, TX 77032"),
        ("Joe Corley Detention Center", "Detention Facility", "Conroe Immigration Court", 
         "806 Hilbig Road, Conroe, TX 77301"),
        ("Polk County Detention Center", "Detention Facility", "Conroe Immigration Court", 
         "806 Hilbig Road, Conroe, TX 77301"),
        ("Eden Detention Center", "Detention Facility", "Conroe Immigration Court", 
         "806 Hilbig Road, Conroe, TX 77301"),
        ("Limestone Detention Center, Groesbeck, TX", "Detention Facility", "Conroe Immigration Court", 
         "806 Hilbig Road, Conroe, TX 77301"),
        ("Houston Contract Detention Facility", "Detention Facility", "Conroe Immigration Court", 
         "806 Hilbig Road, Conroe, TX 77301"),
        
        # Dallas Immigration Court
        ("Dallas, TX - DHS District Office", "DHS Office", "Dallas Immigration Court", 
         "1100 Commerce St. Room 1060, Dallas, TX 75242"),
        ("Catholic Charities, Dallas, TX (Detained Juveniles)", "Juvenile Facility", "Dallas Immigration Court", 
         "1100 Commerce St. Room 1060, Dallas, TX 75242"),
        ("Campus Connections, Dallas, TX (Detained Juveniles)", "Juvenile Facility", "Dallas Immigration Court", 
         "1100 Commerce St. Room 1060, Dallas, TX 75242"),
        
        # El Paso Immigration Court
        ("El Paso, TX - DHS District Office", "DHS Office", "El Paso Immigration Court", 
         "700 East San Antonio Avenue Suite 750, El Paso, TX 79901"),
        ("Reeves County Detention Center", "Detention Facility", "El Paso Immigration Court", 
         "700 East San Antonio Avenue Suite 750, El Paso, TX 79901"),
        ("Paso del Norte, TX Port of Entry", "Port of Entry", "El Paso Immigration Court", 
         "700 East San Antonio Avenue Suite 750, El Paso, TX 79901"),
        ("El Paso Sector Centralized Processing Center", "Processing Center", "El Paso Immigration Court", 
         "700 East San Antonio Avenue Suite 750, El Paso, TX 79901"),
        
        # El Paso SPC Immigration Court
        ("El Paso Service Processing Center", "Service Processing Center", "El Paso SPC Immigration Court", 
         "8915 Montana Ave., Suite 100, El Paso, TX 79925"),
        ("LaTuna Federal Correctional Center", "Federal Correctional Facility", "El Paso SPC Immigration Court", 
         "8915 Montana Ave., Suite 100, El Paso, TX 79925"),
        ("Bluebonnet Detention Center", "Detention Facility", "El Paso SPC Immigration Court", 
         "8915 Montana Ave., Suite 100, El Paso, TX 79925"),
        
        # Harlingen Immigration Court
        ("Harlingen, TX - DHS District Office", "DHS Office", "Harlingen Immigration Court", 
         "2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550"),
        ("Urban Strategies San Benito", "Facility", "Harlingen Immigration Court", 
         "2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550"),
        ("Brownsville International Gateway Bridge", "Port of Entry", "Harlingen Immigration Court", 
         "2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550"),
        ("Hidalgo International Bridge", "Port of Entry", "Harlingen Immigration Court", 
         "2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550"),
        
        # Houston - Jefferson Street Immigration Court
        ("Houston, TX - DHS District Office", "DHS Office", "Houston - Jefferson Street Immigration Court", 
         "500 Jefferson Street, Suite 300, Houston TX 77002"),
        ("Children First Residential Care, Cypress", "Juvenile Facility", "Houston - Jefferson Street Immigration Court", 
         "500 Jefferson Street, Suite 300, Houston TX 77002"),
        ("House of Hope, Baytown", "Juvenile Facility", "Houston - Jefferson Street Immigration Court", 
         "500 Jefferson Street, Suite 300, Houston TX 77002"),
        ("Compass Connections", "Juvenile Facility", "Houston - Jefferson Street Immigration Court", 
         "500 Jefferson Street, Suite 300, Houston TX 77002"),
        
        # Houston - S. Gessner Road Immigration Court
        ("Huntsville Department of Corrections", "Correctional Facility", "Houston - S. Gessner Road Immigration Court", 
         "8701 S. Gessner Road, 10th Floor, Houston TX 77074"),
        
        # Houston - Greenspoint Park Immigration Court
        ("Houston, TX - DHS District Office", "DHS Office", "Houston - Greenspoint Park Immigration Court", 
         "16800 Greenspoint Park Drive, 2nd Floor, Houston, TX 77060"),
        ("Bokenkamp Children's Center", "Juvenile Facility", "Houston - Greenspoint Park Immigration Court", 
         "16800 Greenspoint Park Drive, 2nd Floor, Houston, TX 77060"),
        ("Prairieland Detention Center", "Detention Facility", "Houston - Greenspoint Park Immigration Court", 
         "16800 Greenspoint Park Drive, 2nd Floor, Houston, TX 77060"),
        ("El Valle Detention Facility (Detained CFR)", "Detention Facility", "Houston - Greenspoint Park Immigration Court", 
         "16800 Greenspoint Park Drive, 2nd Floor, Houston, TX 77060"),
        ("Prairieland Detention Facility (Detained CFR)", "Detention Facility", "Houston - Greenspoint Park Immigration Court", 
         "16800 Greenspoint Park Drive, 2nd Floor, Houston, TX 77060"),
        ("Rio Grande Valley Centralized Processing Center", "Processing Center", "Houston - Greenspoint Park Immigration Court", 
         "16800 Greenspoint Park Drive, 2nd Floor, Houston, TX 77060"),
        ("El Valle Detention Center", "Detention Facility", "Houston - Greenspoint Park Immigration Court", 
         "16800 Greenspoint Park Drive, 2nd Floor, Houston, TX 77060"),
        
        # Laredo Immigration Court
        ("San Antonio, TX - DHS District Office", "DHS Office", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Cambria County Prison", "Correctional Facility", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Laredo Port of Entry", "Port of Entry", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Laredo Rio Grande Detention Center", "Detention Facility", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Laredo Detention Center", "Detention Facility", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Laredo Webb County Detention Center", "Detention Facility", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("FCI Big Springs", "Federal Correctional Facility", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Del Rio Port of Entry", "Port of Entry", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Laredo IHF (MPP)", "Hearing Location", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Laredo Processing Center", "Processing Center", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        ("Laredo Sector Centralized Processing Center", "Processing Center", "Laredo Immigration Court", 
         "1406 Jacaman Rd Suite B, Laredo, Tx 78041"),
        
        # Pearsall Immigration Court
        ("South Texas Detention Facility", "Detention Facility", "Pearsall Immigration Court", 
         "566 Veterans Drive, Pearsall, TX 78061"),
        ("South Texas Family Residential Center, Dilley, TX", "Residential Center", "Pearsall Immigration Court", 
         "566 Veterans Drive, Pearsall, TX 78061"),
        ("LaSalle Correctional Center, Cotulla, TX", "Correctional Facility", "Pearsall Immigration Court", 
         "566 Veterans Drive, Pearsall, TX 78061"),
        ("South Texas ICE Processing Center", "Processing Center", "Pearsall Immigration Court", 
         "566 Veterans Drive, Pearsall, TX 78061"),
        ("Karnes County Residential Center, Karnes, TX", "Residential Center", "Pearsall Immigration Court", 
         "566 Veterans Drive, Pearsall, TX 78061"),
        ("Immigration Court Hutto Residential Facility, Taylor, TX", "Residential Center", "Pearsall Immigration Court", 
         "566 Veterans Drive, Pearsall, TX 78061"),
        
        # Port Isabel Immigration Court
        ("Port Isabel Detention Center", "Detention Facility", "Port Isabel Immigration Court", 
         "27991 Buena Vista Blvd., Los Fresnos, TX 78566"),
        ("East Hildago Detention Facility", "Detention Facility", "Port Isabel Immigration Court", 
         "27991 Buena Vista Blvd., Los Fresnos, TX 78566"),
        ("Coastal Bend Detention Center", "Detention Facility", "Port Isabel Immigration Court", 
         "27991 Buena Vista Blvd., Los Fresnos, TX 78566"),
        
        # San Antonio Immigration Court
        ("San Antonio, TX - DHS District Office", "DHS Office", "San Antonio Immigration Court", 
         "800 Dolorosa Street Suite 300, San Antonio, TX 78207"),
        ("Compass Connections TFC, San Antonio, TX", "Facility", "San Antonio Immigration Court", 
         "800 Dolorosa Street Suite 300, San Antonio, TX 78207"),
    ]
    
    return facilities

In [11]:
def scrape_EOIR() -> pd.DataFrame:
    
    # Get judges from judges website
    tx_judges = get_texas_judges()
    
    # Get facilities from admin control website
    facilities = get_texas_facilities()
    
    # Create rows combining facilities and judges
    EOIR_complete = []
    
    # For each Texas court
    for court_name in tx_judges.keys():
        # Get court address (from first facility of this court)
        court_address = ""
        for fac_name, fac_type, fac_court, fac_address in facilities:
            if fac_court == court_name:
                court_address = fac_address
                break
        
        # Get all facilities and judges for this court
        court_facilities = [f for f in facilities if f[2] == court_name]
        court_judges = tx_judges[court_name]
        
        # Add all facilities to rows
        for facility_name, facility_type, _, _ in court_facilities:
            EOIR_complete.append({'Facility_Name': facility_name,
                                  'Facility_Type': facility_type,
                                  'Administrative_Control_Court': court_name,
                                  'Court_Address': court_address,
                                  'Judge_Name': None})

        # Add all judges to rows
        for judge_name in court_judges:
            EOIR_complete.append({'Facility_Name': None,
                                  'Facility_Type': 'Immigration Court',
                                  'Administrative_Control_Court': court_name,
                                  'Court_Address': court_address,
                                  'Judge_Name': judge_name})
    
    df = pd.DataFrame(EOIR_complete)
    
    return df

In [12]:
EOIR_judges_courts_facilities = scrape_EOIR()
display(HTML(f'<div style="height: 300px; overflow: auto; width: fit-content">{EOIR_judges_courts_facilities.to_html()}</div>'))

,Facility_Name,Facility_Type,Administrative_Control_Court,Court_Address,Judge_Name
0,Montgomery Processing Center (CIC),Processing Center,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None
1,Houston Service Processing Center,Service Processing Center,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None
2,Joe Corley Detention Center,Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None
3,Polk County Detention Center,Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None
4,Eden Detention Center,Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None
5,"Limestone Detention Center, Groesbeck, TX",Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None
6,Houston Contract Detention Facility,Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None
7,None,Immigration Court,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",ACIJ Daniel P. Kinnicutt
8,None,Immigration Court,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",Chris Brisack
9,None,Immigration Court,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",Andrew Caborn


### Geocoding Court Addresses from EOIR to Coordinates for Mapping

In [13]:
from geopy.geocoders import ArcGIS

def geocode_arcgis(df):
    geolocator = ArcGIS()
    
    def get_coords(address):
        if pd.isna(address) or address == '':
            return None, None
        try:
            location = geolocator.geocode(address)
            if location:
                return location.latitude, location.longitude
        except Exception as e:
            print(f"Error geocoding {address}: {e}")
            return None, None
        return None, None
    
    df[['latitude', 'longitude']] = df['Court_Address'].apply(lambda x: pd.Series(get_coords(x)))
    
    # Create geometry column from lat/lon
    df['geometry'] = [Point(lon, lat) if pd.notna(lat) and pd.notna(lon) else None
                      for lat, lon in zip(df['latitude'], df['longitude'])]
    
    return df

In [14]:
EOIR_judges_courts_facilities = geocode_arcgis(EOIR_judges_courts_facilities)
EOIR_judges_courts_facilities = gpd.GeoDataFrame(EOIR_judges_courts_facilities, geometry='geometry', crs='EPSG:4326')
display(HTML(f'<div style="height: 300px; overflow: auto; width: fit-content">{EOIR_judges_courts_facilities.to_html()}</div>'))

/cvmfs/cybergis.illinois.edu/software/conda/cybergisx/python3-0.9.0/lib/python3.8/site-packages/pandas/core/dtypes/cast.py:118: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  arr = construct_1d_object_array_from_listlike(values)


,Facility_Name,Facility_Type,Administrative_Control_Court,Court_Address,Judge_Name,latitude,longitude,geometry
0,Montgomery Processing Center (CIC),Processing Center,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None,30.337101,-95.442699,POINT (-95.44270 30.33710)
1,Houston Service Processing Center,Service Processing Center,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None,30.337101,-95.442699,POINT (-95.44270 30.33710)
2,Joe Corley Detention Center,Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None,30.337101,-95.442699,POINT (-95.44270 30.33710)
3,Polk County Detention Center,Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None,30.337101,-95.442699,POINT (-95.44270 30.33710)
4,Eden Detention Center,Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None,30.337101,-95.442699,POINT (-95.44270 30.33710)
5,"Limestone Detention Center, Groesbeck, TX",Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None,30.337101,-95.442699,POINT (-95.44270 30.33710)
6,Houston Contract Detention Facility,Detention Facility,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",None,30.337101,-95.442699,POINT (-95.44270 30.33710)
7,None,Immigration Court,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",ACIJ Daniel P. Kinnicutt,30.337101,-95.442699,POINT (-95.44270 30.33710)
8,None,Immigration Court,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",Chris Brisack,30.337101,-95.442699,POINT (-95.44270 30.33710)
9,None,Immigration Court,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",Andrew Caborn,30.337101,-95.442699,POINT (-95.44270 30.33710)


In [15]:
courts_gdf = EOIR_judges_courts_facilities.groupby('Administrative_Control_Court').first()[['Court_Address', 'latitude', 'longitude', 'geometry']]
courts_gdf.head()

,Court_Address,latitude,longitude,geometry
Administrative_Control_Court,,,,
Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",30.337101,-95.442699,POINT (-95.44270 30.33710)
Dallas Immigration Court,"1100 Commerce St. Room 1060, Dallas, TX 75242",32.778890,-96.802111,POINT (-96.80211 32.77889)
El Paso Immigration Court,"700 East San Antonio Avenue Suite 750, El Paso...",31.759204,-106.482273,POINT (-106.48227 31.75920)
El Paso SPC Immigration Court,"8915 Montana Ave., Suite 100, El Paso, TX 79925",31.793762,-106.369208,POINT (-106.36921 31.79376)
Fort Worth IAC Immigration Court,,NaN,NaN,None


In [16]:
courts_gdf['total_judges'] = EOIR_judges_courts_facilities.groupby('Administrative_Control_Court').agg({'Judge_Name': 'count'})
# courts_gdf['total_s'] = EOIR_judges_courts_facilities.groupby('Administrative_Control_Court').agg({'Judge_Name': 'count'})
courts_gdf.head()
courts_gdf = courts_gdf.sort_values(by = 'total_judges')
courts_gdf = courts_gdf.dropna(subset=['geometry'])
courts_gdf = courts_gdf.reset_index()
display(HTML(f'<div style="height: 300px; overflow: auto; width: fit-content">{courts_gdf.to_html()}</div>'))

,Administrative_Control_Court,Court_Address,latitude,longitude,geometry,total_judges
0,Port Isabel Immigration Court,"27991 Buena Vista Blvd., Los Fresnos, TX 78566",26.158642,-97.358683,POINT (-97.35868 26.15864),4
1,El Paso SPC Immigration Court,"8915 Montana Ave., Suite 100, El Paso, TX 79925",31.793762,-106.369208,POINT (-106.36921 31.79376),5
2,Harlingen Immigration Court,"2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550",26.194727,-97.718684,POINT (-97.71868 26.19473),5
3,Laredo Immigration Court,"1406 Jacaman Rd Suite B, Laredo, Tx 78041",27.562795,-99.471972,POINT (-99.47197 27.56280),6
4,Pearsall Immigration Court,"566 Veterans Drive, Pearsall, TX 78061",28.896570,-99.120292,POINT (-99.12029 28.89657),7
5,El Paso Immigration Court,"700 East San Antonio Avenue Suite 750, El Paso, TX 79901",31.759204,-106.482273,POINT (-106.48227 31.75920),8
6,Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",30.337101,-95.442699,POINT (-95.44270 30.33710),10
7,Dallas Immigration Court,"1100 Commerce St. Room 1060, Dallas, TX 75242",32.778890,-96.802111,POINT (-96.80211 32.77889),11
8,Houston - Jefferson Street Immigration Court,"500 Jefferson Street, Suite 300, Houston TX 77002",29.752928,-95.373800,POINT (-95.37380 29.75293),13
9,San Antonio Immigration Court,"800 Dolorosa Street Suite 300, San Antonio, TX 78207",29.423911,-98.498824,POINT (-98.49882 29.42391),13


In [17]:
courts_gdf.to_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/courts_gdf.geojson", driver='GeoJSON')

In [18]:
EOIR_judges_courts_facilities.to_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/EOIR_complete_gdf.geojson", driver='GeoJSON')

### 287(g) County Participation Data

In [19]:
tx_287g = pd.read_csv('/home/jovyan/work/GGIS 407/Final Project/data/participating_pendingAgencies02122026pm.csv', encoding='utf-8', encoding_errors='ignore')
tx_287g['COUNTY'] = tx_287g['COUNTY'].str.replace(' County', '', regex=False)
tx_287g['SIGNED'] = pd.to_datetime(tx_287g['SIGNED'], errors='coerce')
tx_287g.head()

,STATE,LAW ENFORCEMENT AGENCY,TYPE,COUNTY,SUPPORT TYPE,SIGNED,MOA,ADDENDUM
0,TEXAS,Alice Police Department,Municipality,Jim Wells,Task Force Model,2025-12-17,link,NaN
1,TEXAS,Anderson County Sheriff's Office,County,Anderson,Warrant Service Officer,2025-06-18,link,NaN
2,TEXAS,Andrews County Sheriff's Office,County,Andrews,Warrant Service Officer,2025-11-14,link,NaN
3,TEXAS,Angelina County Sheriff's Office,County,Angelina,Warrant Service Officer,2025-06-18,link,NaN
4,TEXAS,Aransas County Sheriff's Office,County,Aransas,Jail Enforcement Model,2020-06-08,link,link


In [20]:
counties = gpd.read_file('/home/jovyan/work/GGIS 407/Final Project/data/geojson-counties-fips.json')
tx_counties = counties[counties['STATE'] == '48']
tx_counties = tx_counties[['id', 'NAME', 'CENSUSAREA', 'geometry']]
tx_counties = tx_counties.rename(columns={'id':'FIPS', 'NAME' : 'COUNTY'})
tx_counties.head()

,FIPS,COUNTY,CENSUSAREA,geometry
525,48357,Ochiltree,917.627,"POLYGON ((-100.93606 36.49960, -100.91851 36.4..."
526,48359,Oldham,1500.533,"POLYGON ((-103.04238 35.18316, -103.04250 35.2..."
527,48367,Parker,903.478,"POLYGON ((-97.54418 32.99418, -97.55037 32.580..."
528,48379,Rains,229.452,"POLYGON ((-95.66539 32.96043, -95.63502 32.720..."
529,48381,Randall,911.544,"POLYGON ((-102.16884 34.74742, -102.16747 35.1..."


In [21]:
tx_287g_counties_gdf = tx_counties.merge(tx_287g, on = ['COUNTY'])
tx_287g_counties_gdf = tx_287g_counties_gdf.sort_values(by=['COUNTY', 'SIGNED'])
tx_287g_counties_gdf.head()
display(HTML(f'<div style="height: 300px; overflow: auto; width: fit-content">{tx_287g_counties_gdf.to_html()}</div>'))

,FIPS,COUNTY,CENSUSAREA,geometry,STATE,LAW ENFORCEMENT AGENCY,TYPE,SUPPORT TYPE,SIGNED,MOA,ADDENDUM
74,48001,Anderson,1062.602,"POLYGON ((-95.42851 32.08447, -95.44675 31.84312, -95.41291 31.83516, -95.25886 31.60996, -95.27320 31.59289, -95.65176 31.54179, -95.73928 31.50406, -95.71011 31.61559, -95.78730 31.61838, -95.79408 31.66031, -95.86126 31.68745, -95.87594 31.75550, -95.98057 31.78456, -95.99413 31.86626, -96.02779 31.87824, -96.02295 31.95758, -96.06217 31.95634, -96.05279 32.00590, -95.42851 32.08447))",TEXAS,Anderson County Sheriff's Office,County,Warrant Service Officer,2025-06-18,link,NaN
183,48005,Angelina,797.778,"POLYGON ((-94.32662 31.22475, -94.12963 31.09928, -94.45782 31.03333, -94.56194 31.05895, -94.84295 31.14658, -94.90950 31.33706, -94.95811 31.38693, -94.95942 31.38888, -94.96625 31.39120, -94.96452 31.39556, -94.96763 31.39741, -94.96937 31.39695, -94.97358 31.39976, -94.97778 31.39938, -94.97607 31.40200, -94.97629 31.40525, -94.97936 31.40598, -94.97603 31.40774, -94.98305 31.41159, -94.98475 31.41385, -94.98806 31.41442, -94.99004 31.41336, -94.99383 31.41422, -94.99411 31.41784, -94.99713 31.41678, -94.99825 31.42004, -95.00126 31.41795, -95.00557 31.42135, -95.00334 31.42571, -95.00021 31.42820, -94.86586 31.52692, -94.72868 31.45723, -94.54489 31.43172, -94.53063 31.39865, -94.49587 31.40573, -94.32662 31.22475))",TEXAS,Angelina County Sheriff's Office,County,Warrant Service Officer,2025-06-18,link,NaN
11,48011,Armstrong,909.109,"POLYGON ((-101.47156 34.74746, -101.62926 34.74765, -101.62940 34.75006, -101.62294 35.18312, -101.08628 35.18214, -101.09075 34.74825, -101.47156 34.74746))",TEXAS,Armstrong County Sheriff's Office,County,Task Force Model,2026-01-26,link,NaN
75,48013,Atascosa,1219.544,"POLYGON ((-98.80084 28.64749, -98.80490 29.09043, -98.80476 29.25069, -98.40734 29.11444, -98.19099 28.88233, -98.09831 28.78695, -98.33503 28.61266, -98.33505 28.64828, -98.80085 28.64731, -98.80084 28.64749))",TEXAS,Atascosa County Sheriff's Office,County,Task Force Model,2025-05-08,link,NaN
76,48015,Austin,646.508,"POLYGON ((-96.62198 30.04428, -96.29285 30.09615, -96.14605 30.07022, -96.08454 30.00514, -96.12275 29.96854, -96.12131 29.84196, -96.04923 29.80319, -96.03271 29.72794, -96.02485 29.60288, -96.08891 29.60166, -96.17542 29.63381, -96.25923 29.66891, -96.34448 29.83015, -96.41328 29.82499, -96.56984 29.96152, -96.62198 30.04428))",TEXAS,Austin County Sheriff's Office,County,Warrant Service Officer,2025-04-16,link,NaN
77,48017,Bailey,826.797,"POLYGON ((-103.04398 34.31275, -102.61515 34.31289, -102.61545 33.82512, -103.04735 33.82467, -103.04691 33.85030, -103.04564 33.90154, -103.04570 33.90630, -103.04489 33.94562, -103.04395 33.97463, -103.04362 34.00363, -103.04353 34.01801, -103.04355 34.03271, -103.04375 34.03729, -103.04377 34.04154, -103.04372 34.04232, -103.04377 34.04355, -103.04374 34.04999, -103.04369 34.06308, -103.04352 34.07938, -103.04357 34.08795, -103.04364 34.25690, -103.04372 34.28944, -103.04394 34.30259, -103.04398 34.31275))",TEXAS,Bailey County Sheriff's Office,County,Warrant Service Officer,2025-07-31,link,NaN
118,48027,Bell,1051.016,"POLYGON ((-97.07019 30.98622, -97.25908 30.88960, -97.31551 30.75237, -97.62529 30.87043, -97.82851 30.90619, -97.84037 30.92932, -97.91168 31.03492, -97.91385 31.06588, -97.90710 31.06937, -97.41861 31.32020, -97.34343 31.24422, -97.27811 31.27980, -97.07019 30.98622))",TEXAS,Holland Police Department,Municipality,Task Force Model,2025-11-04,link,NaN
80,48029,Bexar,1239.820,"POLYGON ((-98.28702 29.25763, -98.40734 29.11444, -98.80476 29.25069, -98.80655 29.69071, -98.77878 29.72017, -98.64612 29.74518, -98.55049 29.76071, -98.44385 29.71965, -98.33862 29.72179, -98.37807 29.66261, -98.31093 29.59447, -98.29852 29.56114, -98.27292 29.55088, -98.12677 29.48225, -98.13417 29.44175, -98.13719 29.44043, -98.28702 29.25763))",TEXAS,Bexar County Sheriff's Office,County,Warrant Service Officer,2025-10-17,link,NaN
79,48029,Bexar,1239.820,"POLY

In [22]:
print(tx_287g_counties_gdf.columns.tolist())
print(tx_287g_counties_gdf['SUPPORT TYPE'].unique())

['FIPS', 'COUNTY', 'CENSUSAREA', 'geometry', 'STATE', 'LAW ENFORCEMENT AGENCY', 'TYPE', 'SUPPORT TYPE', 'SIGNED', 'MOA', 'ADDENDUM']
['Warrant Service Officer' 'Task Force Model' 'Jail Enforcement Model'
 'Jail Enforcement Model ']


In [23]:
tx_287g_counties_gdf['SUPPORT TYPE'] = tx_287g_counties_gdf['SUPPORT TYPE'].replace('Jail Enforcement Model ', 'Jail Enforcement Model')
print(tx_287g_counties_gdf['SUPPORT TYPE'].unique())

['Warrant Service Officer' 'Task Force Model' 'Jail Enforcement Model']


In [24]:
print(tx_287g_counties_gdf.groupby('LAW ENFORCEMENT AGENCY')['SUPPORT TYPE'].count().sort_values(ascending=False)) 
print(tx_287g_counties_gdf.groupby('COUNTY')['SUPPORT TYPE'].count().sort_values(ascending=False)) 

LAW ENFORCEMENT AGENCY
Kerr County Sheriff's Office       3
Winkler County Sheriff's Office    3
Victoria County Sheriffs Office    3
Dimmit County Sheriff's Office     3
Refugio County Sheriffs Office     3
                                  ..
Jack County Sheriff's Office       1
Bullard Police Department          1
Jefferson Police Department        1
Brown County Sheriff's Office      1
Alice Police Department            1
Name: SUPPORT TYPE, Length: 178, dtype: int64
COUNTY
Galveston    8
Bexar        5
Refugio      4
Smith        4
Jim Wells    4
            ..
Angelina     1
Kinney       1
Kenedy       1
Kaufman      1
Anderson     1
Name: SUPPORT TYPE, Length: 146, dtype: int64


In [25]:
pivot_287g = tx_287g_counties_gdf.pivot_table(index=['LAW ENFORCEMENT AGENCY'], columns='SUPPORT TYPE', values='FIPS', aggfunc='count', fill_value=0)
display(HTML(f'<div style="height: 300px; overflow: auto; width: fit-content">{pivot_287g.to_html()}</div>'))
print(len(pivot_287g))
print(len(tx_287g_counties_gdf))

SUPPORT TYPE,Jail Enforcement Model,Task Force Model,Warrant Service Officer
LAW ENFORCEMENT AGENCY,,,
Alice Police Department,0,1,0
Anderson County Sheriff's Office,0,0,1
Angelina County Sheriff's Office,0,0,1
Armstrong County Sheriff's Office,0,1,0
Atascosa County Sheriff's Office,0,1,0
Austin County Sheriff's Office,0,0,1
Bailey County Sheriff's Office,0,0,1
Balcones Heights Police Department,0,1,0
Bexar County Constable Precinct 3,0,1,0


178
217


In [26]:
gdf_287g = tx_287g_counties_gdf.drop_duplicates('LAW ENFORCEMENT AGENCY').drop(columns='SUPPORT TYPE')
tx_287g_counties_gdf = gdf_287g.merge(pivot_287g, on='LAW ENFORCEMENT AGENCY')

display(HTML(f'<div style="height: 300px; overflow: auto; width: fit-content">{tx_287g_counties_gdf.to_html()}</div>'))
tx_287g_counties_gdf = tx_287g_counties_gdf.set_crs('EPSG:4326', allow_override=True)
tx_287g_counties_gdf.to_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/tx_287g_counties_gdf.geojson", driver='GeoJSON')

,FIPS,COUNTY,CENSUSAREA,geometry,STATE,LAW ENFORCEMENT AGENCY,TYPE,SIGNED,MOA,ADDENDUM,Jail Enforcement Model,Task Force Model,Warrant Service Officer
0,48001,Anderson,1062.602,"POLYGON ((-95.42851 32.08447, -95.44675 31.84312, -95.41291 31.83516, -95.25886 31.60996, -95.27320 31.59289, -95.65176 31.54179, -95.73928 31.50406, -95.71011 31.61559, -95.78730 31.61838, -95.79408 31.66031, -95.86126 31.68745, -95.87594 31.75550, -95.98057 31.78456, -95.99413 31.86626, -96.02779 31.87824, -96.02295 31.95758, -96.06217 31.95634, -96.05279 32.00590, -95.42851 32.08447))",TEXAS,Anderson County Sheriff's Office,County,2025-06-18,link,NaN,0,0,1
1,48005,Angelina,797.778,"POLYGON ((-94.32662 31.22475, -94.12963 31.09928, -94.45782 31.03333, -94.56194 31.05895, -94.84295 31.14658, -94.90950 31.33706, -94.95811 31.38693, -94.95942 31.38888, -94.96625 31.39120, -94.96452 31.39556, -94.96763 31.39741, -94.96937 31.39695, -94.97358 31.39976, -94.97778 31.39938, -94.97607 31.40200, -94.97629 31.40525, -94.97936 31.40598, -94.97603 31.40774, -94.98305 31.41159, -94.98475 31.41385, -94.98806 31.41442, -94.99004 31.41336, -94.99383 31.41422, -94.99411 31.41784, -94.99713 31.41678, -94.99825 31.42004, -95.00126 31.41795, -95.00557 31.42135, -95.00334 31.42571, -95.00021 31.42820, -94.86586 31.52692, -94.72868 31.45723, -94.54489 31.43172, -94.53063 31.39865, -94.49587 31.40573, -94.32662 31.22475))",TEXAS,Angelina County Sheriff's Office,County,2025-06-18,link,NaN,0,0,1
2,48011,Armstrong,909.109,"POLYGON ((-101.47156 34.74746, -101.62926 34.74765, -101.62940 34.75006, -101.62294 35.18312, -101.08628 35.18214, -101.09075 34.74825, -101.47156 34.74746))",TEXAS,Armstrong County Sheriff's Office,County,2026-01-26,link,NaN,0,1,0
3,48013,Atascosa,1219.544,"POLYGON ((-98.80084 28.64749, -98.80490 29.09043, -98.80476 29.25069, -98.40734 29.11444, -98.19099 28.88233, -98.09831 28.78695, -98.33503 28.61266, -98.33505 28.64828, -98.80085 28.64731, -98.80084 28.64749))",TEXAS,Atascosa County Sheriff's Office,County,2025-05-08,link,NaN,0,1,0
4,48015,Austin,646.508,"POLYGON ((-96.62198 30.04428, -96.29285 30.09615, -96.14605 30.07022, -96.08454 30.00514, -96.12275 29.96854, -96.12131 29.84196, -96.04923 29.80319, -96.03271 29.72794, -96.02485 29.60288, -96.08891 29.60166, -96.17542 29.63381, -96.25923 29.66891, -96.34448 29.83015, -96.41328 29.82499, -96.56984 29.96152, -96.62198 30.04428))",TEXAS,Austin County Sheriff's Office,County,2025-04-16,link,NaN,0,0,1
5,48017,Bailey,826.797,"POLYGON ((-103.04398 34.31275, -102.61515 34.31289, -102.61545 33.82512, -103.04735 33.82467, -103.04691 33.85030, -103.04564 33.90154, -103.04570 33.90630, -103.04489 33.94562, -103.04395 33.97463, -103.04362 34.00363, -103.04353 34.01801, -103.04355 34.03271, -103.04375 34.03729, -103.04377 34.04154, -103.04372 34.04232, -103.04377 34.04355, -103.04374 34.04999, -103.04369 34.06308, -103.04352 34.07938, -103.04357 34.08795, -103.04364 34.25690, -103.04372 34.28944, -103.04394 34.30259, -103.04398 34.31275))",TEXAS,Bailey County Sheriff's Office,County,2025-07-31,link,NaN,0,0,1
6,48027,Bell,1051.016,"POLYGON ((-97.07019 30.98622, -97.25908 30.88960, -97.31551 30.75237, -97.62529 30.87043, -97.82851 30.90619, -97.84037 30.92932, -97.91168 31.03492, -97.91385 31.06588, -97.90710 31.06937, -97.41861 31.32020, -97.34343 31.24422, -97.27811 31.27980, -97.07019 30.98622))",TEXAS,Holland Police Department,Municipality,2025-11-04,link,NaN,0,1,0
7,48029,Bexar,1239.820,"POLYGON ((-98.28702 29.25763, -98.40734 29.11444, -98.80476 29.25069, -98.80655 29.69071, -98.77878 29.72017, -98.64612 29.74518, -98.55049 29.76071, -98.44385 29.71965, -98.33862 29.72179, -98.37807 29.66261, -98.31093 29.59447, -98.29852 29.56114, -98.27292 29.55088, -98.12677 29.48225, -98.13417 29.44175, -98.13719 29.44043, -98.28702 29.25763))",TEXAS,Bexar County Sheriff's Office,County,2025-10-17,link,NaN,0,0,1
8,48029,Bexar,1239.820,"POLYGON ((-98.28702 29.25763, -98.40734 29.11444, -98.80476 29.25069, -98.80655 29.6907

In [27]:
tx_dps_contracts_llb = pd.read_csv('/home/jovyan/work/GGIS 407/Final Project/data/download_contracts.csv')
tx_dps_contracts_llb.head()

,Agency,Award Date,Vendor,Contract,Contract-ID,Subject,Current Contract Value,Status,Completion Date,Procurement Method,Category,NGIP Codes and Categories,Vendor ID
0,Department of Public Safety,2004-04-01,PARKER COUNTY,405-12-20534,<a href='https://cms.lbb.texas.gov/Default.asp...,Weatherford Building Lease 10647,"$129,600",Active,03/31/2031,Competitive,Goods,971-45 Rental or Lease,"""17560011094001"""
1,Department of Public Safety,2004-04-12,"DATAMAXX APPLIED TECHNOLOGIES, INC.",405-15-P001977,<a href='https://cms.lbb.texas.gov/Default.asp...,Datamaxx,"$13,072,639",Completed,03/19/2019,Competitive,Information,920-45 Information Technology,"""15930816788*00"""
2,Department of Public Safety,2004-07-01,TARANTINO PROPERTIES INC,405-12-20538,<a href='https://cms.lbb.texas.gov/Default.asp...,Building Lease - Pasadena 20003,"$529,907",Completed,06/30/2019,Competitive,Goods,971-45 Rental or Lease,"""17421287735003"""
3,Department of Public Safety,2004-07-01,TOTAL ENERGY INVESTMENTS LLC,405-13-31801,<a href='https://cms.lbb.texas.gov/Default.asp...,Building Lease - Pearland 20038,"$1,214,695",Active,07/31/2027,Competitive,Goods,971-45 Rental or Lease,"""14615121515000"""
4,Department of Public Safety,2004-10-01,BLUMIN-HIGHPOINT LTD,405-12-20544,<a href='https://cms.lbb.texas.gov/Default.asp...,Building Lease - Dallas,"$830,811",Completed,05/31/2017,Competitive,Goods,971-45 Rental or Lease,"""11418463565000"""


### PostGIS Processing & Query Operations

#### Initialize PostGIS and Import Processed Datasets and GeoDataFrames Into Neew PostGIS DB

In [3]:
os.getcwd()

'/home/jovyan/work/GGIS 407/Final Project'

In [4]:
os.listdir("/home/jovyan/work/GGIS 407/Final Project/data")

['facilities.csv',
 'G_drive_harlingen.graphml',
 'geojson-counties-fips.json',
 'texas-260312.osm.pbf',
 'EOIR scraper.ipynb',
 'pgdata',
 '.ipynb_checkpoints',
 'ice-aor-county-shp.shx',
 'tx_287g_counties_gdf.geojson',
 'Untitled.ipynb',
 'immigration_court_administrative_control.csv',
 'G_drive_elpaso_proj.graphml',
 'ice-aor-county-shp.prj',
 'participating_pendingAgencies02122026pm.csv',
 'courts_gdf.geojson',
 'G_drive_harlingen_proj.graphml',
 'G_drive_elpaso.graphml',
 'download_contracts.csv',
 'processed',
 'ice-aor-county-shp.shp',
 'texas_immigration_judges.csv',
 'ice-aor-county-shp.dbf',
 'EOIR_complete_gdf.geojson',
 'EFF_towers_01_26.csv']

In [5]:
os.environ["PGDATA"] = "/home/jovyan/work/GGIS 407/Final Project/data/pgdata"
!pg_ctl initdb
!pg_ctl start
%reload_ext sql
!createdb tx_immigration_enforcement
%sql postgresql://localhost:5432/tx_immigration_enforcement

The files belonging to this database system will be owned by user "jovyan".
This user must also own the server process.

The database cluster will be initialized with locale "en_US.UTF-8".
The default database encoding has accordingly been set to "UTF8".
The default text search configuration will be set to "english".

Data page checksums are disabled.

initdb: error: directory "/home/jovyan/work/GGIS 407/Final Project/data/pgdata" exists but is not empty
If you want to create a new database system, either remove or empty
the directory "/home/jovyan/work/GGIS 407/Final Project/data/pgdata" or run initdb
with an argument other than "/home/jovyan/work/GGIS 407/Final Project/data/pgdata".
pg_ctl: database system initialization failed
waiting for server to start....2026-03-14 03:50:00.923 UTC [11828] LOG:  starting PostgreSQL 13.5 on x86_64-conda-linux-gnu, compiled by x86_64-conda-linux-gnu-cc (GCC) 9.4.0, 64-bit
2026-03-14 03:50:00.927 UTC [11828] LOG:  could not bind IPv6 address "::1": 

In [6]:
%%sql
drop extension if exists postgis cascade;
create extension postgis;

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
Done.


[]

In [7]:
engine = create_engine("postgresql://localhost:5432/tx_immigration_enforcement")

In [8]:
!shp2pgsql -s 4269 -d -I "/home/jovyan/work/GGIS 407/Final Project/data/ice-aor-county-shp.shp" public.ice_aor | psql -q -d tx_immigration_enforcement

Field aland is an FTDouble with width 24 and precision 15
Field awater is an FTDouble with width 24 and precision 15
Shapefile type: Polygon
Postgis type: MULTIPOLYGON[2]
ERROR:  column not found in geometry_columns table
CONTEXT:  PL/pgSQL function dropgeometrycolumn(character varying,character varying,character varying,character varying) line 33 at RAISE
SQL statement "SELECT public.DropGeometryColumn('',$1,$2,$3)"
PL/pgSQL function dropgeometrycolumn(character varying,character varying,character varying) line 5 at SQL statement
                    addgeometrycolumn                    
---------------------------------------------------------
 public.ice_aor.geom SRID:4269 TYPE:MULTIPOLYGON DIMS:2 
(1 row)



In [9]:
# Import GeoJSON files into PostgreSQL database
!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/courts_gdf.geojson" \
    -nln public.courts \
    -t_srs EPSG:4269 \
    -overwrite

!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/EOIR_complete_gdf.geojson" \
    -nln public.detention_centers \
    -t_srs EPSG:4269 \
    -overwrite

!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/tx_287g_counties_gdf.geojson" \
    -nln public.counties_287g \
    -t_srs EPSG:4269 \
    -overwrite

!ogr2ogr -f "PostgreSQL" PG:"dbname=tx_immigration_enforcement host=localhost port=5432" \
    "/home/jovyan/work/GGIS 407/Final Project/data/processed/towers_gdf.geojson" \
    -nln public.surveillance_towers \
    -t_srs EPSG:4269 \
    -overwrite

In [10]:
%sql \d

 * postgresql://localhost:5432/tx_immigration_enforcement
19 rows affected.


Schema,Name,Type,Owner
public,aor_courts,table,jovyan
public,aor_to_counties,table,jovyan
public,aor_towers,table,jovyan
public,counties_287g,table,jovyan
public,counties_287g_ogc_fid_seq,sequence,jovyan
public,court_tower_aor_matching,table,jovyan
public,courts,table,jovyan
public,courts_ogc_fid_seq,sequence,jovyan
public,detention_centers,table,jovyan
public,detention_centers_ogc_fid_seq,sequence,jovyan


In [11]:
%%sql
SELECT column_name, data_type 
FROM information_schema.columns 
WHERE table_name = 'detention_centers';

 * postgresql://localhost:5432/tx_immigration_enforcement
9 rows affected.


column_name,data_type
ogc_fid,integer
facility_name,character varying
facility_type,character varying
administrative_control_court,character varying
court_address,character varying
judge_name,character varying
latitude,double precision
longitude,double precision
wkb_geometry,USER-DEFINED


In [12]:
%%sql
select *
from geometry_columns

 * postgresql://localhost:5432/tx_immigration_enforcement
5 rows affected.


f_table_catalog,f_table_schema,f_table_name,f_geometry_column,coord_dimension,srid,type
tx_immigration_enforcement,public,ice_aor,geom,2,4269,MULTIPOLYGON
tx_immigration_enforcement,public,courts,wkb_geometry,2,4269,POINT
tx_immigration_enforcement,public,detention_centers,wkb_geometry,2,4269,POINT
tx_immigration_enforcement,public,counties_287g,wkb_geometry,2,4269,GEOMETRY
tx_immigration_enforcement,public,surveillance_towers,wkb_geometry,2,4269,POINT


#### Create AOR to County Polygon Bounds

In [13]:
%%sql
drop table if exists aor_to_counties;
create table aor_to_counties as
select 
    name AS county, 
    aor_nam, 
    offc_nm, 
    geom as county_geom, 
    aland as land_area_m2, 
    awater as water_area_m2
from ice_aor
where state_n = 'Texas'
order by aor_nam

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
254 rows affected.


[]

In [14]:
%sql select distinct(aor_nam) from aor_to_counties

 * postgresql://localhost:5432/tx_immigration_enforcement
5 rows affected.


aor_nam
Houston
El Paso
San Antonio
Dallas
Harlingen


In [15]:
%sql select distinct(aor_nam), count(county) from aor_to_counties group by aor_nam

 * postgresql://localhost:5432/tx_immigration_enforcement
5 rows affected.


aor_nam,count
San Antonio,36
Harlingen,15
Dallas,128
El Paso,18
Houston,57


In [16]:
%sql select * from aor_to_counties limit 5

 * postgresql://localhost:5432/tx_immigration_enforcement
5 rows affected.


county,aor_nam,offc_nm,county_geom,land_area_m2,water_area_m2
Crockett,Dallas,Dallas,...,7270935061.0000000000000,53329.000000000000000
Scurry,Dallas,Dallas,0106000020AD1000000100000001030000000100000029000000E97FB9162D4B59C0B7B75B920350404044334FAE294B59C0C93EC8B2605040409D47C5FF1D4B59C098874CF910704040AFCC5B751D4B59C046EBA86A827A404023F609A0184B59C009C38025577B4040698B6B7C264659C08D7C5EF1D47B404074EACA67794259C07C613255307C40408658FD11864159C09204E10A287C4040F27A30293E4059C02849D74CBE7B4040B80721205F3F59C0A471A8DF857B4040B97020240B3959C09BC761307F7B4040A7CCCD37A23859C087A3AB74777B40405C1B2AC6F92959C0DCDAC2F3527B40406B44300E2E2A59C0349F73B7EB5D4040CB4C69FD2D2A59C08066101FD85D4040132D793C2D2A59C0AF963B33C15A40406B2A8BC22E2A59C06BF294D574574040DD257156442A59C038A0A52BD84E4040475A2A6F472A59C0F9F6AE415F44404017D68D77472A59C06D00362042444040C93846B2472A59C02219726C3D43404020D099B4A92C59C0F3C81F0C3C4340406536C824233259C0BBB7223141434040996379573D3359C0B91B446B45434040A4E36A64573459C0D49810734943404082531F48DE3959C086014BAE6243404020D26F5F073B59C04C88B9A46A4340401250E108523B59C0F2086EA46C4340406919A9F7543B59C0F2086EA46C43404086C8E9EBF93B59C0A2CF471971434040C47AA356983C59C0A0336953754340402B1895D4093E59C06C43C5387F434040FFCEF6E80D3F59C0683F52448643404027A3CA30EE3F59C0FAB836548C4340403E05C078064059C0EB6E9EEA904340405CC64D0D344259C044520B25934340400F7BA180ED4A59C0EE0A7DB08C434040F54718062C4B59C024809BC58B4340408065A549294B59C0CC28965B5A4F40402B89EC832C4B59C0593673486A4F4040E97FB9162D4B59C0B7B75B9203504040,2345090475.0000000000000,5436957.000000000000000
Briscoe,Dallas,Dallas,0106000020AD1000000100000001030000000100000025000000BB809719365E59C0C2DB8310903741402A3BFDA02E5E59C0FEEF880AD54341407099D365315E59C03E247CEF6F444140F4C5DE8B2F5E59C06CCEC133A14F41405B41D3122B5E59C0A3B08BA207504140DE054A0A2C5E59C001DE02098A534140F54718062C5E59C00A4B3CA06C564140540262122E5E59C05A80B6D5AC5F414093399677D55B59C02B306475AB5F4140635E471CB25859C00F0A4AD1CA5F4140F9872D3D9A5159C0B85A272EC75F4140399D64ABCB4C59C0508BC1C3B45F414043C9E4D4CE4559C071AE6186C65F4140C26D6DE1793C59C0ED9BFBABC75F4140CC99ED0A7D3C59C05532005471514140EF6FD05E7D3C59C0B6D617096D5141409624CFF57D3C59C077BAF3C4735041401A51DA1B7C3C59C0560E2DB29D4F414047054EB6813C59C04419AA622A3F41407BF83251843C59C07E1EA33CF33E4140DE3B6A4C883C59C002D53F88643E414017670C73823C59C04B0169FF033E4140BE6C3B6D8D3C59C0CC43A67C0828414073F6CE68AB3D59C0BAD8B45208284140F31E679AB03D59C03271AB2006284140AAD381ACA74259C06CEA3C2AFE274140BCB20B06D74959C021E9D32AFA274140A515DF50F84A59C027DA5548F9274140959A3DD00A4B59C0F3CCCB61F7274140DBDD03745F4C59C00AF5F411F827414080F0A1444B4E59C0040473F4F8274140D9CF6229925059C052D50451F72741408670CCB2275159C04D69FD2D01284140E3C798BB965159C06440F67AF7274140A75CE15D2E5E59C0156F641EF9274140C9CA2F83315E59C072A8DF85AD2F4140BB809719365E59C0C2DB831090374140,2330991073.0000000000000,4068657.000000000000000
Reagan,Dallas,Dallas,0106000020AD100000010000000103000000010000001300000069FE98D6A67159C00BB6114F761F3F40FA264D83A27159C0ADF6B0170A203F40253CA1D79F7159C023A12DE7525C3F40BEC0AC50A47159C09D0DF967065D3F40F88A6EBDA67159C070438CD7BCA63F40B282DF86185159C08578245E9EA63F40D1E97937165159C0475854C4E9A43F406C5B94D9205159C07D772B4B748E3F40ED832C0B265159C0A3C9C51858873F40EE224C512E5159C0CF49EF1B5F7B3F4094895B05315159C0E5637781927A3F40FAB660A92E5159C0C3D5011077793F40520B2593535159C08B51D7DAFB583F40310BED9C665159C03CFA5FAE454B3F404E7E8B4E965159C03BFDA02E52143F406E895C70065859C0C685032159143F40EB1ED95C355D59C0E481C8224D143F40C51C041DAD7159C07BBE66B96C143F4069FE98D6A67159C00BB6114F761F3F40,3044049119.0000000000000,1792716.000000000000000
Bowie,Dallas,Dallas,...,2291982022.0000000000000,98385599.000000000000000


#### Explore towers_gdf/surveillance_towers table

In [17]:
%sql select * from surveillance_towers limit 5

 * postgresql://localhost:5432/tx_immigration_enforcement
5 rows affected.


ogc_fid,tower name,current tower type,tower subcategory,sector,sector acronym,station/aor,vendor,wkb_geometry
1,BRP Customs B&M,Remote Video Surveillance System (RVSS),Relocatable Tower,Rio Grande Valley Sector,RGV,Brownsville Station,General Dynamics,0101000020AD100000AC394030476058C057ED9A90D6E43940
2,BRP Mulberry,Remote Video Surveillance System (RVSS),Relocatable Tower,Rio Grande Valley Sector,RGV,Brownsville Station,General Dynamics,0101000020AD100000C537143E5B6658C0401361C3D3F73940
3,DRT-DRS-02 Upper Weir2,Remote Video Surveillance System (RVSS),Monopole Tower,Del Rio Sector,DRT,Del Rio Station,General Dynamics,0101000020AD1000003F00A94D9C4259C014967840D96C3D40
4,DRT-DRS-06 Lower Weir Dam2,Remote Video Surveillance System (RVSS),Monopole Tower,Del Rio Sector,DRT,Del Rio Station,None,0101000020AD100000AFEB17EC863B59C007B64AB038543D40
5,DRT-DRS-07 POE Bridge,Remote Video Surveillance System (RVSS),Monopole Tower,Del Rio Sector,DRT,Del Rio Station,None,0101000020AD100000739D465A2A3B59C0959F54FB74543D40


In [18]:
%sql select count(distinct wkb_geometry) from surveillance_towers

 * postgresql://localhost:5432/tx_immigration_enforcement
1 rows affected.


count
292


In [19]:
%sql select distinct("current tower type"),  count("current tower type") from surveillance_towers group by "current tower type"

 * postgresql://localhost:5432/tx_immigration_enforcement
3 rows affected.


current tower type,count
Autonomous Surveillance Tower (AST),109
Uncategorized,11
Remote Video Surveillance System (RVSS),173


In [20]:
%sql select distinct("current tower type"),  "tower subcategory", count("tower name") from surveillance_towers group by "current tower type", "tower subcategory"

 * postgresql://localhost:5432/tx_immigration_enforcement
6 rows affected.


current tower type,tower subcategory,count
Remote Video Surveillance System (RVSS),Monopole Tower,80
Autonomous Surveillance Tower (AST),Extended Range Sentry Tower,2
Remote Video Surveillance System (RVSS),Relocatable Tower,60
Remote Video Surveillance System (RVSS),None,33
Autonomous Surveillance Tower (AST),None,107
Uncategorized,None,11


In [21]:
%%sql 
select 
    sector, 
    "station/aor", 
    count("tower name") as tower_count 
from surveillance_towers 
group by sector, "station/aor" 
order by sector, tower_count asc

 * postgresql://localhost:5432/tx_immigration_enforcement
21 rows affected.


sector,station/aor,tower_count
Big Bend Sector,None,49
Del Rio Sector,Eagle Pass Station,2
Del Rio Sector,Del Rio Station,9
Del Rio Sector,None,33
El Paso Sector,Ysleta Station,1
El Paso Sector,None,7
El Paso Sector,El Paso Station,20
Laredo Sector,Laredo South Station,3
Laredo Sector,Laredo West Station,5
Laredo Sector,None,61


In [22]:
%%sql
select 
    sector, 
    "tower subcategory", 
    vendor, 
    count("tower name") as tower_count 
from surveillance_towers 
group by sector, "tower subcategory", vendor
order by sector, tower_count

 * postgresql://localhost:5432/tx_immigration_enforcement
28 rows affected.


sector,tower subcategory,vendor,tower_count
Big Bend Sector,None,Teledyne Flir,1
Big Bend Sector,None,None,1
Big Bend Sector,None,Anduril,47
Del Rio Sector,None,None,1
Del Rio Sector,None,General Dynamics,2
Del Rio Sector,Monopole Tower,None,3
Del Rio Sector,Monopole Tower,General Dynamics,18
Del Rio Sector,None,Anduril,20
El Paso Sector,Relocatable Tower,General Dynamics,1
El Paso Sector,None,General Dynamics,2


#### Create table: `detention_facilities`

In [23]:
%%sql 
drop table if exists detention_facilities;
create table detention_facilities as
select 
    facility_name, 
    facility_type, 
    administrative_control_court, 
    court_address, 
    ST_AsText(wkb_geometry) as wkb_geometry
FROM detention_centers
WHERE facility_name IS NOT NULL

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
55 rows affected.


[]

In [24]:
%%sql
with tbl as (
    select 
        administrative_control_court, 
        count(facility_name) as total_facility_count
    from detention_facilities 
    group by administrative_control_court
)

select 
    d.administrative_control_court, 
    d.facility_type, 
    count(d.facility_name) as facility_type_count, 
    tbl.total_facility_count
from detention_facilities as d
left join tbl on d.administrative_control_court = tbl.administrative_control_court 
group by d.administrative_control_court, d.facility_type, tbl.total_facility_count
order by d.administrative_control_court

 * postgresql://localhost:5432/tx_immigration_enforcement
36 rows affected.


administrative_control_court,facility_type,facility_type_count,total_facility_count
Conroe Immigration Court,Detention Facility,5,7
Conroe Immigration Court,Processing Center,1,7
Conroe Immigration Court,Service Processing Center,1,7
Dallas Immigration Court,DHS Office,1,3
Dallas Immigration Court,Juvenile Facility,2,3
El Paso Immigration Court,Detention Facility,1,4
El Paso Immigration Court,DHS Office,1,4
El Paso Immigration Court,Port of Entry,1,4
El Paso Immigration Court,Processing Center,1,4
El Paso SPC Immigration Court,Detention Facility,1,3


#### Create table: `judge_details`

In [25]:
%%sql 
select 
    administrative_control_court, 
    court_address, 
    total_judges,
    ST_AsText(wkb_geometry), 
    wkb_geometry
FROM courts 

 * postgresql://localhost:5432/tx_immigration_enforcement
12 rows affected.


administrative_control_court,court_address,total_judges,st_astext,wkb_geometry
Port Isabel Immigration Court,"27991 Buena Vista Blvd., Los Fresnos, TX 78566",4,POINT(-97.358682873261 26.158641827366),0101000020AD100000A59202A9F45658C08D5034C09C283A40
El Paso SPC Immigration Court,"8915 Montana Ave., Suite 100, El Paso, TX 79925",5,POINT(-106.369208351881 31.793762008564),0101000020AD1000004B2F111CA1975AC0F396ABFC33CB3F40
Harlingen Immigration Court,"2009 West Jefferson Ave., Suite 300, Harlingen, TX 78550",5,POINT(-97.718684038948 26.194727004596),0101000020AD10000016DC56EBFE6D58C04B6304A1D9313A40
Laredo Immigration Court,"1406 Jacaman Rd Suite B, Laredo, Tx 78041",6,POINT(-99.471972072251 27.562795340811),0101000020AD10000061BC59CA34DE58C0DA1FFF5A13903B40
Pearsall Immigration Court,"566 Veterans Drive, Pearsall, TX 78061",7,POINT(-99.120291973986 28.896570461288),0101000020AD1000006F8F1BDDB2C758C0A6CA49A485E53C40
El Paso Immigration Court,"700 East San Antonio Avenue Suite 750, El Paso, TX 79901",8,POINT(-106.482273268738 31.7592035475),0101000020AD100000BE3DB390DD9E5AC00A85E7295BC23F40
Conroe Immigration Court,"806 Hilbig Road, Conroe, TX 77301",10,POINT(-95.442699014881 30.337100979098),0101000020AD100000A9B83F2E55DC57C0EDACF03F4C563E40
Dallas Immigration Court,"1100 Commerce St. Room 1060, Dallas, TX 75242",11,POINT(-96.802110689134 32.778890064349),0101000020AD100000906612C8553358C076C76CABB2634040
Houston - Jefferson Street Immigration Court,"500 Jefferson Street, Suite 300, Houston TX 77002",13,POINT(-95.373800357512 29.75292813487),0101000020AD100000D1AF5558ECD757C04381F3E5BFC03D40
San Antonio Immigration Court,"800 Dolorosa Street Suite 300, San Antonio, TX 78207",13,POINT(-98.498824480222 29.423911194049),0101000020AD100000DA3F83BDEC9F58C049D9AA71856C3D40


In [26]:
%%sql 
drop table if exists judge_details;
create table judge_details as
select 
    judge_name, 
    administrative_control_court, 
    court_address, 
    ST_AsText(wkb_geometry) as geom_astext, 
    wkb_geometry
FROM detention_centers
WHERE judge_name IS NOT NULL

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
131 rows affected.


[]

In [27]:
%%sql
select 
    administrative_control_court, 
    count(judge_name) as judge_count, 
    geom_astext
from judge_details
group by administrative_control_court, geom_astext
order by judge_count desc

 * postgresql://localhost:5432/tx_immigration_enforcement
13 rows affected.


administrative_control_court,judge_count,geom_astext
Fort Worth IAC Immigration Court,18,None
Houston - Greenspoint Park Immigration Court,17,POINT(-95.402484068369 29.945864458543)
Houston - S. Gessner Road Immigration Court,14,POINT(-95.528444794718 29.686902919638)
San Antonio Immigration Court,13,POINT(-98.498824480222 29.423911194049)
Houston - Jefferson Street Immigration Court,13,POINT(-95.373800357512 29.75292813487)
Dallas Immigration Court,11,POINT(-96.802110689134 32.778890064349)
Conroe Immigration Court,10,POINT(-95.442699014881 30.337100979098)
El Paso Immigration Court,8,POINT(-106.482273268738 31.7592035475)
Pearsall Immigration Court,7,POINT(-99.120291973986 28.896570461288)
Laredo Immigration Court,6,POINT(-99.471972072251 27.562795340811)


#### Create table: `county_287g`

In [28]:
%%sql 
select 
    county,
    "law enforcement agency", 
    signed, 
    "jail enforcement model", 
    "task force model", 	
    "warrant service officer",
    ST_AsText(wkb_geometry), 
    wkb_geometry as geom
from counties_287g
limit 10

 * postgresql://localhost:5432/tx_immigration_enforcement
10 rows affected.


county,law enforcement agency,signed,jail enforcement model,task force model,warrant service officer,st_astext,geom
Anderson,Anderson County Sheriff's Office,2025-06-18 00:00:00+00:00,0,0,1,"POLYGON((-95.428512 32.084475,-95.446747 31.843116,-95.412908 31.835157,-95.258859 31.609959,-95.273203 31.592886,-95.651764 31.541791,-95.739279 31.504056,-95.710112 31.615587,-95.7873 31.618385,-95.794081 31.66031,-95.861262 31.687451,-95.875937 31.755503,-95.980568 31.784561,-95.994127 31.866258,-96.027788 31.878242,-96.022955 31.957581,-96.062172 31.95634,-96.052786 32.005895,-95.428512 32.084475))",0103000020AD1000000100000013000000637C98BD6CDB57C0302AA913D00A404083A5BA8097DC57C0FDBB3E73D6D73F406F10AD156DDA57C0850662D9CCD53F409ED1562591D057C09FE6E445269C3F40D28A6F287CD157C09A417C60C7973F40732D5A80B6E957C06344A2D0B28A3F40A7E7DD5850EF57C0425A63D009813F40D11F9A7972ED57C0BFD7101C979D3F4003098A1F63F257C04356B77A4E9E3F4000581D39D2F257C0C5387F130AA93F4067D2A6EA1EF757C03D9AEAC9FCAF3F40CE16105A0FF857C0A70705A568C13F4044E048A0C1FE57C091B75CFDD8C83F408544DAC69FFF57C0FBE59315C3DD3F4029CE5147C70158C040F9BB77D4E03F40E6913F18780158C05F45460724F53F4086AE44A0FA0358C04DDBBFB2D2F43F40F2EB87D8600358C0DF1AD82AC1004040637C98BD6CDB57C0302AA913D00A4040
Angelina,Angelina County Sheriff's Office,2025-06-18 00:00:00+00:00,0,0,1,"POLYGON((-94.326616 31.224754,-94.129632 31.09928,-94.45782 31.033326,-94.561943 31.058952,-94.842947 31.146578,-94.909502 31.337059,-94.95811 31.38693,-94.959415 31.388884,-94.966254 31.391205,-94.964521 31.395558,-94.967634 31.397412,-94.969369 31.396948,-94.973581 31.399759,-94.97778 31.399381,-94.976068 31.402,-94.976291 31.40525,-94.979364 31.405975,-94.976033 31.407744,-94.983053 31.411593,-94.984753 31.41385,-94.988061 31.414417,-94.990043 31.413356,-94.993832 31.41422,-94.994108 31.417835,-94.997132 31.41678,-94.998247 31.42004,-95.001258 31.417949,-95.005566 31.421349,-95.003345 31.42571,-95.000212 31.428196,-94.865857 31.526916,-94.728679 31.457226,-94.544888 31.431715,-94.530634 31.398654,-94.495874 31.405728,-94.326616 31.224754))",0103000020AD10000001000000240000006C96CB46E79457C02BA5677A89393F40F72004E44B8857C09A25016A6A193F401ADD41EC4C9D57C0431B800D88083F40D5CDC5DFF6A357C0A708707A170F3F40B950F9D7F2B557C02750C42286253F406269E04735BA57C08DD2A57F49563F401EFE9AAC51BD57C05ED72FD80D633F40AE122C0E67BD57C00BF0DDE68D633F404568041BD7BD57C01C08C90226643F4089D349B6BABD57C0D2AB014A43653F40DA1F28B7EDBD57C0B709F7CABC653F40663046240ABE57C06D3656629E653F4072C0AE264FBE57C01B48179B56663F40B4AB90F293BE57C0CFA44DD53D663F4004ABEAE577BE57C0F4FDD478E9663F4044183F8D7BBE57C039B4C876BE673F404EB857E6ADBE57C05DFE43FAED673F4046B41D5377BE57C0E62329E961683F40054F2157EABE57C03543AA285E693F40D368723106BF57C01973D712F2693F40FF5C34643CBF57C00CE8853B176A3F408FA850DD5CBF57C0FA9AE5B2D1693F40F86D88F19ABF57C0605969520A6A3F40492C29779FBF57C0C91F0C3CF76A3F40E272BC02D1BF57C07BA01518B26A3F4021956247E3BF57C00803CFBD876B3F40F0366F9C14C057C05A65A6B4FE6A3F400DFE7E315BC057C0CA332F87DD6B3F40BA66F2CD36C057C085949F54FB6C3F404644317903C057C02C47C8409E6D3F40D0807A336AB757C038BD8BF7E3863F400C923EADA2AE57C082E15CC30C753F40E7FEEA71DFA257C05131CEDF846E3F4053094FE8F5A157C06A6B44300E663F40F33B4D66BC9F57C04D124BCADD673F406C96CB46E79457C02BA5677A89393F40
Armstrong,Armstrong County Sheriff's Office,2026-01-26 00:00:00+00:00,0,1,0,"POLYGON((-101.471562 34.747462,-101.629257 34.747649,-101.629396 34.750056,-101.622941 35.183117,-101.086281 35.18214,-101.090749 34.748246,-101.471562 34.747462))",0103000020AD1000000100000007000000540262122E5E59C05A80B6D5AC5F4140DCF126BF456859C08DF161F6B25F4140ED0E2906486859C09415C3D5016041409A95ED43DE6759C0BB2BBB60709741400951BEA0854559C08FA50F5D5097414043C9E4D4CE4559C071AE6186C65F4140540262122E5E59C05A80B6D5AC5F4140
Atascosa,Atascosa County Sheriff's Office,2025-05-08 00:00:00+00:00,0,1,0,"POLYGON((-98.800841 28.647487,-98.8049 29.090434,-98.804763 29.250693,-98.407336 29.114435,-98.190991 28.882333,-98.098315 28.786949,-98.335031 28.612658,-98.335047 28

## Import Area of Responsibility Shapefiles from Deportation Data Project at UC Berkeley

In [29]:
aor = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/ice-aor-county-shp.shp")

In [30]:
aor.head()

,STATEFP,COUNTYF,COUNTYN,GEOIDFQ,GEOID,NAME,NAMELSA,STUSPS,STATE_N,LSAD,ALAND,AWATER,offc_nm,notes,aor_nam,geometry
0,06,079,00277304,0500000US06079,06079,San Luis Obispo,San Luis Obispo County,CA,California,06,8.549141e+09,815650229.0,Los Angeles,None,Los Angeles,"POLYGON ((-121.34636 35.79518, -121.24498 35.7..."
1,06,081,00277305,0500000US06081,06081,San Mateo,San Mateo County,CA,California,06,1.161921e+09,757163219.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-122.52085 37.59418, -122.51533 37.5..."
2,06,091,00277310,0500000US06091,06091,Sierra,Sierra County,CA,California,06,2.468695e+09,23299110.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-121.05748 39.53999, -121.05643 39.5..."
3,06,067,00277298,0500000US06067,06067,Sacramento,Sacramento County,CA,California,06,2.500063e+09,75323439.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-121.86254 38.06795, -121.86160 38.0..."
4,06,107,00277318,0500000US06107,06107,Tulare,Tulare County,CA,California,06,1.249384e+10,37260863.0,San Francisco,Counties in Northern California not in the LA ...,San Francisco,"POLYGON ((-119.56647 36.49434, -119.56366 36.4..."


In [31]:
print(aor.columns.tolist())
print(aor.crs)

['STATEFP', 'COUNTYF', 'COUNTYN', 'GEOIDFQ', 'GEOID', 'NAME', 'NAMELSA', 'STUSPS', 'STATE_N', 'LSAD', 'ALAND', 'AWATER', 'offc_nm', 'notes', 'aor_nam', 'geometry']
epsg:4269


In [32]:
texas_aor = aor[aor['STATE_N'] == 'Texas'] 

In [33]:
print(texas_aor['aor_nam'].unique())
print("Counties Per Each Area of Responsibility (AOR): ")
print(texas_aor.groupby('aor_nam')['NAMELSA'].count()) 

['Houston' 'Dallas' 'Harlingen' 'El Paso' 'San Antonio']
Counties Per Each Area of Responsibility (AOR): 
aor_nam
Dallas         128
El Paso         18
Harlingen       15
Houston         57
San Antonio     36
Name: NAMELSA, dtype: int64


In [36]:
facilities = pd.read_csv("/home/jovyan/work/GGIS 407/Final Project/data/facilities.csv")
courts = pd.read_csv("/home/jovyan/work/GGIS 407/Final Project/data/texas_immigration_judges.csv")

print(facilities.columns.tolist())
print(courts.columns.tolist())

['detention_facility_code', 'detention_facility_name', 'address', 'city', 'county', 'state', 'zip', 'aor', 'latitude', 'longitude', 'type_detailed', 'type_grouped']
['Court', 'Judge_Name', 'Initials', 'WebEx_Link', 'Access_Code']


In [37]:
detention_centers = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/EOIR_complete_gdf.geojson")
counties_287g_enforcement_types = gpd.read_file("/home/jovyan/work/GGIS 407/Final Project/data/processed/tx_287g_counties_gdf.geojson")                

# Now get road networks for each AOR within buffer zone of points of interest nodes

## Dissolve County Polygons into Corresponding AOR Polygons from AOR Shapefiles

In [34]:
G_drive_harlingen = texas_aor[texas_aor['aor_nam'] == 'Harlingen']
harlingen_union_poly = G_drive_harlingen.geometry.unary_union

G_drive_eptx = texas_aor[texas_aor['aor_nam'] == 'El Paso']
el_paso_union_poly = G_drive_eptx.geometry.unary_union

G_drive_satx = texas_aor[texas_aor['aor_nam'] == 'San Antonio']
san_antonio_union_poly = G_drive_satx.geometry.unary_union

G_drive_houston = texas_aor[texas_aor['aor_nam'] == 'Houston']
houston_union_poly = G_drive_houston.geometry.unary_union

G_drive_dallas = texas_aor[texas_aor['aor_nam'] == 'Dallas']
dallas_union_poly = G_drive_dallas.geometry.unary_union

## Harlingen OSMnx Street Network

In [34]:
# G_drive_harlingen = ox.load_graphml('/home/jovyan/work/GGIS 407/Final Project/data/G_drive_harlingen.graphml')
# G_drive_harlingen_proj = ox.project_graph(G_drive_harlingen, to_crs="EPSG:3857")
# ox.save_graphml(G_drive_harlingen_proj, filepath="data/G_drive_harlingen_proj.graphml")
G_drive_harlingen_proj = ox.load_graphml('/home/jovyan/work/GGIS 407/Final Project/data/G_drive_harlingen_proj.graphml')

2026-03-14 03:07:07 Converting node, edge, and graph-level attribute data types
2026-03-14 03:07:26 Loaded graph with 95166 nodes and 254578 edges from "/home/jovyan/work/GGIS 407/Final Project/data/G_drive_harlingen_proj.graphml"


In [35]:
nodes_harlingen_drive_proj, edges_harlingen_drive_proj = ox.graph_to_gdfs(G_drive_harlingen_proj)

2026-03-14 03:09:10 Created nodes GeoDataFrame from graph
2026-03-14 03:09:25 Created edges GeoDataFrame from graph


### Harlingen Spatial Predicate Database Operations (Match Courts/Towers to AOR)

In [36]:
%%sql
drop table if exists aor_towers;
create table aor_towers as
with aor as (
    select 
        aor_nam as aor_name, 
        ST_Union(county_geom) as aor_geom
    from aor_to_counties
    group by aor_nam
)
select 
    aor.aor_name, 
    t."station/aor",
    t."current tower type", 
    t."tower subcategory",
    t.wkb_geometry as tower_loc, 
    aor.aor_geom
from aor
left join surveillance_towers AS t on ST_Within(t.wkb_geometry, aor.aor_geom)

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
294 rows affected.


[]

In [37]:
%%sql
drop table if exists aor_courts;
create table aor_courts as
with aor as (
    select 
        aor_nam as aor_name, 
        ST_Union(county_geom) as aor_geom
    from aor_to_counties
    group by aor_nam
)
select 
    aor.aor_name, 
    c.administrative_control_court, 
    c.wkb_geometry as court_loc, 
    aor.aor_geom
from aor
left join courts AS c on ST_Within(c.wkb_geometry, aor.aor_geom)

 * postgresql://localhost:5432/tx_immigration_enforcement
Done.
12 rows affected.


[]

### Find and retrieve nearest nodes for immigration courts, facilities, and towers on Harlingen OSMnx network graph:

In [38]:
court_nodes = gpd.read_postgis("SELECT * FROM aor_courts", engine, geom_col="court_loc")
court_nodes = court_nodes.to_crs(epsg=3857)

tower_nodes = gpd.read_postgis("SELECT * FROM aor_towers", engine, geom_col="tower_loc")
tower_nodes = tower_nodes.to_crs(epsg=3857)
tower_nodes = tower_nodes[tower_nodes.geometry.notna()].copy()

In [39]:
court_nodes_harlingen = court_nodes[court_nodes["aor_name"] == "Harlingen"].copy()
court_nodes_harlingen["nearest_node"] = court_nodes_harlingen.geometry.apply(lambda geom: ox.distance.nearest_nodes(G_drive_harlingen_proj, X=geom.x, Y=geom.y))

2026-03-14 03:10:01 Created nodes GeoDataFrame from graph
2026-03-14 03:10:02 Created nodes GeoDataFrame from graph
2026-03-14 03:10:03 Created nodes GeoDataFrame from graph


In [40]:
court_nodes_harlingen

,aor_name,administrative_control_court,court_loc,aor_geom,nearest_node
3,Harlingen,Port Isabel Immigration Court,POINT (-10837919.002 3018742.717),0106000020AD1000001A00000001030000000100000074...,222836077
4,Harlingen,Harlingen Immigration Court,POINT (-10877994.148 3023218.774),0106000020AD1000001A00000001030000000100000074...,302698583
5,Harlingen,Laredo Immigration Court,POINT (-11073169.279 3193963.273),0106000020AD1000001A00000001030000000100000074...,297387250


In [41]:
tower_nodes_harlingen = tower_nodes[tower_nodes["aor_name"] == "Harlingen"].copy()
tower_nodes_harlingen["nearest_node"] = tower_nodes_harlingen.geometry.apply(lambda geom: ox.distance.nearest_nodes(G_drive_harlingen_proj, X=geom.x, Y=geom.y))

2026-03-14 03:10:09 Created nodes GeoDataFrame from graph
2026-03-14 03:10:09 Created nodes GeoDataFrame from graph
2026-03-14 03:10:11 Created nodes GeoDataFrame from graph
2026-03-14 03:10:11 Created nodes GeoDataFrame from graph
2026-03-14 03:10:13 Created nodes GeoDataFrame from graph
2026-03-14 03:10:13 Created nodes GeoDataFrame from graph
2026-03-14 03:10:15 Created nodes GeoDataFrame from graph
2026-03-14 03:10:17 Created nodes GeoDataFrame from graph
2026-03-14 03:10:17 Created nodes GeoDataFrame from graph
2026-03-14 03:10:18 Created nodes GeoDataFrame from graph
2026-03-14 03:10:18 Created nodes GeoDataFrame from graph
2026-03-14 03:10:20 Created nodes GeoDataFrame from graph
2026-03-14 03:10:20 Created nodes GeoDataFrame from graph
2026-03-14 03:10:22 Created nodes GeoDataFrame from graph
2026-03-14 03:10:23 Created nodes GeoDataFrame from graph
2026-03-14 03:10:24 Created nodes GeoDataFrame from graph
2026-03-14 03:10:26 Created nodes GeoDataFrame from graph
2026-03-14 03:

In [47]:
len(tower_nodes_harlingen)

167

In [48]:
tower_nodes_harlingen.head(20)

,aor_name,station/aor,current tower type,tower subcategory,tower_loc,aor_geom,nearest_node
48,Harlingen,Brownsville Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10854134.036 2985945.804),0106000020AD1000001A00000001030000000100000074...,222986226
49,Harlingen,Brownsville Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10864706.493 2995127.433),0106000020AD1000001A00000001030000000100000074...,223022822
50,Harlingen,Fort Brown Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10839899.390 2983474.870),0106000020AD1000001A00000001030000000100000074...,7100071413
51,Harlingen,Fort Brown Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10843694.828 2982435.026),0106000020AD1000001A00000001030000000100000074...,223001251
52,Harlingen,Fort Brown Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10846376.900 2981995.079),0106000020AD1000001A00000001030000000100000074...,10253850155
53,Harlingen,Fort Brown Station,Remote Video Surveillance System (RVSS),Monopole Tower,POINT (-10847301.325 2982163.549),0106000020AD1000001A00000001030000000100000074...,10253850155
54,Harlingen,Harlingen Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10877351.830 3004531.429),0106000020AD1000001A00000001030000000100000074...,223062549
55,Harlingen,Harlingen Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10885348.751 3005791.544),0106000020AD1000001A00000001030000000100000074...,11780996263
56,Harlingen,Harlingen Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10897195.754 3007365.191),0106000020AD1000001A00000001030000000100000074...,12899655211
57,Harlingen,Harlingen Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10873337.984 3003937.585),0106000020AD1000001A00000001030000000100000074...,11306555194


In [49]:
tower_nodes_harlingen["node_lon"] = tower_nodes_harlingen["nearest_node"].apply(lambda n: G_drive_harlingen_proj.nodes[n]["x"])
tower_nodes_harlingen["node_lat"] = tower_nodes_harlingen["nearest_node"].apply(lambda n: G_drive_harlingen_proj.nodes[n]["y"])

tower_nodes_harlingen["node_geom"] = tower_nodes_harlingen.apply(lambda row: Point(row["node_lon"], row["node_lat"]), axis=1)
tower_nodes_harlingen["node_distance_m"] = tower_nodes_harlingen.apply(lambda row: row["tower_loc"].distance(row["node_geom"]), axis=1)

/cvmfs/cybergis.illinois.edu/software/conda/cybergisx/python3-0.9.0/lib/python3.8/site-packages/pandas/core/dtypes/cast.py:118: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  arr = construct_1d_object_array_from_listlike(values)


In [50]:
tower_nodes_harlingen.sort_values(by = 'node_distance_m', ascending = False)

,aor_name,station/aor,current tower type,tower subcategory,tower_loc,aor_geom,nearest_node,node_lon,node_lat,node_geom,node_distance_m
146,Harlingen,None,Autonomous Surveillance Tower (AST),None,POINT (-11117435.581 3225881.261),0106000020AD1000001A00000001030000000100000074...,229175328,-1.111450e+07,3.223439e+06,POINT (-11114502.86609125 3223438.63818633),3816.703412
204,Harlingen,None,Remote Video Surveillance System (RVSS),None,POINT (-11152709.140 3273222.686),0106000020AD1000001A00000001030000000100000074...,229262428,-1.114953e+07,3.271541e+06,POINT (-11149529.98914732 3271540.568892543),3596.737245
211,Harlingen,None,Remote Video Surveillance System (RVSS),None,POINT (-11044394.796 3091259.776),0106000020AD1000001A00000001030000000100000074...,230154978,-1.104343e+07,3.094347e+06,POINT (-11043428.04288753 3094347.378940837),3235.413903
88,Harlingen,None,Remote Video Surveillance System (RVSS),Monopole Tower,POINT (-11085440.909 3199172.913),0106000020AD1000001A00000001030000000100000074...,229291175,-1.108613e+07,3.202217e+06,POINT (-11086128.0844855 3202216.616072895),3120.310846
87,Harlingen,None,Remote Video Surveillance System (RVSS),Monopole Tower,POINT (-11085440.909 3199172.913),0106000020AD1000001A00000001030000000100000074...,229291175,-1.108613e+07,3.202217e+06,POINT (-11086128.0844855 3202216.616072895),3120.310846
...,...,...,...,...,...,...,...,...,...,...,...
162,Harlingen,Harlingen Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10880094.186 3003746.787),0106000020AD1000001A00000001030000000100000074...,12162169024,-1.088008e+07,3.003730e+06,POINT (-10880077.5549307 3003730.036761438),23.604503
179,Harlingen,None,Autonomous Surveillance Tower (AST),None,POINT (-10820980.000 2995029.265),0106000020AD1000001A00000001030000000100000074...,222966794,-1.082100e+07,2.995029e+06,POINT (-10820998.052513 2995029.490708954),18.054088
105,Harlingen,Weslaco Station,Remote Video Surveillance System (RVSS),Relocatable Tower,POINT (-10926290.397 3009265.142),0106000020AD1000001A00000001030000000100000074...,151335509,-1.092631e+07,3.009264e+06,POINT (-10926307.8492763 3009264.167435856),17.479837
121,Harlingen,Harlingen Station,Remote Video Surveillance System (RVSS),Monopole Tower,POINT (-10888741.707 3007487.630),0106000020AD1000001A00000001030000000100000074...,8258192635,-1.088873e+07,3.007497e+06,POINT (-10888730.24083888 3007496.651569367),14.589727


In [51]:
tower_nodes_harlingen['node_distance_m'].describe()

count     167.000000
mean      742.706126
std       806.022502
min         9.970562
25%       150.456240
50%       488.180109
75%      1029.102713
max      3816.703412
Name: node_distance_m, dtype: float64

In [42]:
court_nodes_harlingen.drop(columns=['aor_geom', 'node_geom'], errors='ignore').to_file(f"/home/jovyan/work/GGIS 407/Final Project/data/processed/court_nodes_harlingen.geojson", driver='GeoJSON')
tower_nodes_harlingen.drop(columns=['aor_geom', 'node_geom'], errors='ignore').to_file(f"/home/jovyan/work/GGIS 407/Final Project/data/processed/tower_nodes_harlingen.geojson", driver='GeoJSON')

## El Paso OSMnx Street Network

In [52]:
# G_drive_elpaso = ox.graph_from_polygon(el_paso_union_poly, network_type='drive')
# G_drive_elpaso = ox.add_edge_speeds(G_drive_elpaso)
# G_drive_elpaso = ox.add_edge_travel_times(G_drive_elpaso)
# ox.save_graphml(G_drive_elpaso, filepath="data/G_drive_elpaso.graphml")
# print("Done!")
G_drive_elpaso = ox.load_graphml('/home/jovyan/work/GGIS 407/Final Project/data/G_drive_elpaso.graphml')

2026-03-14 02:21:37 Converting node, edge, and graph-level attribute data types
2026-03-14 02:21:48 Loaded graph with 67876 nodes and 184162 edges from "/home/jovyan/work/GGIS 407/Final Project/data/G_drive_elpaso.graphml"


In [53]:
# G_drive_elpaso_proj = ox.project_graph(G_drive_elpaso, to_crs = "EPSG:3857")
# ox.save_graphml(G_drive_elpaso_proj, filepath="data/G_drive_elpaso_proj.graphml")
G_drive_elpaso_proj = ox.load_graphml('/home/jovyan/work/GGIS 407/Final Project/data/G_drive_elpaso_proj.graphml')

2026-03-14 02:22:14 Converting node, edge, and graph-level attribute data types
2026-03-14 02:22:28 Loaded graph with 67876 nodes and 184162 edges from "/home/jovyan/work/GGIS 407/Final Project/data/G_drive_elpaso_proj.graphml"


nodes_eptx_drive_proj, edges_eptx_drive_proj = ox.graph_to_gdfs(G_drive_elpaso_proj)

### Find and retrieve nearest nodes for immigration courts, facilities, and towers on El Paso OSMnx network graph:

In [54]:
court_nodes_eptx = court_nodes[court_nodes["aor_name"] == "El Paso"].copy()
court_nodes_eptx["nearest_node"] = court_nodes_eptx.geometry.apply(lambda geom: ox.distance.nearest_nodes(G_drive_elpaso_proj, X=geom.x, Y=geom.y))

2026-03-14 02:22:28 Created nodes GeoDataFrame from graph
2026-03-14 02:22:30 Created nodes GeoDataFrame from graph


In [55]:
court_nodes_eptx

,aor_name,administrative_control_court,court_loc,aor_geom,nearest_node
6,El Paso,El Paso SPC Immigration Court,POINT (-11840966.110 3736269.028),0103000020AD10000001000000B30F0000B51B7DCC07F8...,11044394717
7,El Paso,El Paso Immigration Court,POINT (-11853552.439 3731743.689),0103000020AD10000001000000B30F0000B51B7DCC07F8...,179325035


In [56]:
tower_nodes_eptx = tower_nodes[tower_nodes["aor_name"] == "El Paso"].copy()
tower_nodes_eptx["nearest_node"] = tower_nodes_eptx.geometry.apply(lambda geom: ox.distance.nearest_nodes(G_drive_elpaso_proj, X=geom.x, Y=geom.y))

2026-03-14 02:22:31 Created nodes GeoDataFrame from graph
2026-03-14 02:22:31 Created nodes GeoDataFrame from graph
2026-03-14 02:22:31 Created nodes GeoDataFrame from graph
2026-03-14 02:22:32 Created nodes GeoDataFrame from graph
2026-03-14 02:22:34 Created nodes GeoDataFrame from graph
2026-03-14 02:22:34 Created nodes GeoDataFrame from graph
2026-03-14 02:22:34 Created nodes GeoDataFrame from graph
2026-03-14 02:22:35 Created nodes GeoDataFrame from graph
2026-03-14 02:22:35 Created nodes GeoDataFrame from graph
2026-03-14 02:22:37 Created nodes GeoDataFrame from graph
2026-03-14 02:22:37 Created nodes GeoDataFrame from graph
2026-03-14 02:22:38 Created nodes GeoDataFrame from graph
2026-03-14 02:22:38 Created nodes GeoDataFrame from graph
2026-03-14 02:22:38 Created nodes GeoDataFrame from graph
2026-03-14 02:22:41 Created nodes GeoDataFrame from graph
2026-03-14 02:22:41 Created nodes GeoDataFrame from graph
2026-03-14 02:22:41 Created nodes GeoDataFrame from graph
2026-03-14 02:

In [57]:
len(tower_nodes_eptx)

78

In [58]:
tower_nodes_eptx["node_lon"] = tower_nodes_eptx["nearest_node"].apply(lambda n: G_drive_elpaso_proj.nodes[n]["x"])
tower_nodes_eptx["node_lat"] = tower_nodes_eptx["nearest_node"].apply(lambda n: G_drive_elpaso_proj.nodes[n]["y"])

tower_nodes_eptx["node_geom"] = tower_nodes_eptx.apply(lambda row: Point(row["node_lon"], row["node_lat"]), axis=1)
tower_nodes_eptx["node_distance_m"] = tower_nodes_eptx.apply(lambda row: row["tower_loc"].distance(row["node_geom"]), axis=1)

/cvmfs/cybergis.illinois.edu/software/conda/cybergisx/python3-0.9.0/lib/python3.8/site-packages/pandas/core/dtypes/cast.py:118: ShapelyDeprecationWarning: The array interface is deprecated and will no longer work in Shapely 2.0. Convert the '.coords' to a numpy array instead.
  arr = construct_1d_object_array_from_listlike(values)


In [59]:
tower_nodes_eptx.sort_values(by = 'node_distance_m', ascending = False)

,aor_name,station/aor,current tower type,tower subcategory,tower_loc,aor_geom,nearest_node,node_lon,node_lat,node_geom,node_distance_m
245,El Paso,None,Autonomous Surveillance Tower (AST),None,POINT (-11634192.521 3463129.186),0103000020AD10000001000000B30F0000B51B7DCC07F8...,152262082,-1.164071e+07,3.475678e+06,POINT (-11640713.99525324 3475678.180457391),14142.379436
244,El Paso,None,Autonomous Surveillance Tower (AST),None,POINT (-11631164.753 3458573.997),0103000020AD10000001000000B30F0000B51B7DCC07F8...,9783049022,-1.162138e+07,3.448556e+06,POINT (-11621378.59007083 3448556.252604476),14004.434733
240,El Paso,None,Autonomous Surveillance Tower (AST),None,POINT (-11626924.104 3454450.423),0103000020AD10000001000000B30F0000B51B7DCC07F8...,9783049022,-1.162138e+07,3.448556e+06,POINT (-11621378.59007083 3448556.252604476),8092.834175
289,El Paso,None,Autonomous Surveillance Tower (AST),None,POINT (-11359597.082 3496401.165),0103000020AD10000001000000B30F0000B51B7DCC07F8...,228817312,-1.136570e+07,3.501612e+06,POINT (-11365695.53083721 3501611.918651951),8021.410809
274,El Paso,None,Autonomous Surveillance Tower (AST),None,POINT (-11637254.297 3470029.780),0103000020AD10000001000000B30F0000B51B7DCC07F8...,152262082,-1.164071e+07,3.475678e+06,POINT (-11640713.99525324 3475678.180457391),6623.740958
...,...,...,...,...,...,...,...,...,...,...,...
261,El Paso,None,Autonomous Surveillance Tower (AST),None,POINT (-11790943.934 3677015.301),0103000020AD10000001000000B30F0000B51B7DCC07F8...,148541578,-1.179098e+07,3.677026e+06,POINT (-11790977.2629347 3677025.895017019),34.972180
219,El Paso,El Paso Station,Remote Video Surveillance System (RVSS),Monopole Tower,POINT (-11856295.477 3732207.925),0103000020AD10000001000000B30F0000B51B7DCC07F8...,4583368319,-1.185628e+07,3.732239e+06,POINT (-11856284.85709135 3732238.830540296),32.679290
223,El Paso,El Paso Station,Remote Video Surveillance System (RVSS),Monopole Tower,POINT (-11849396.518 3732160.876),0103000020AD10000001000000B30F0000B51B7DCC07F8...,179671568,-1.184940e+07,3.732189e+06,POINT (-11849403.97672644 3732189.287400925),29.373941
227,El Paso,El Paso Station,Remote Video Surveillance System (RVSS),Monopole Tower,POINT (-11841716.587 3727591.063),0103000020AD10000001000000B30F0000B51B7DCC07F8...,179233874,-1.184173e+07,3.727604e+06,POINT (-11841734.39776926 3727603.888809532),21.948659


In [60]:
tower_nodes_eptx['node_distance_m'].describe()

count       78.000000
mean      2020.405855
std       2821.750661
min         19.036447
25%        132.489551
50%        829.053221
75%       3295.166747
max      14142.379436
Name: node_distance_m, dtype: float64

In [ ]:
court_nodes_eptx.drop(columns=['aor_geom', 'node_geom'], errors='ignore').to_file(f"/home/jovyan/work/GGIS 407/Final Project/data/processed/court_nodes_el_paso.geojson", driver='GeoJSON')
tower_nodes_eptx.drop(columns=['aor_geom', 'node_geom'], errors='ignore').to_file(f"/home/jovyan/work/GGIS 407/Final Project/data/processed/tower_nodes_el_paso.geojson", driver='GeoJSON')

In [61]:
# nearest = min(poi_points, key=lambda p: p.distance(nodes_drive.loc[origin_node].geometry))
# distance_m = nearest.distance(nodes_drive.loc[origin_node].geometry)
# nearest_rest_name = restaurants.loc[restaurants.geometry == nearest, 'name'].values[0]

# route_distance = ox.shortest_path(G_drive_harlingen, origin_node, dest_node, weight='length')
# route_time = ox.shortest_path(G_drive_harlingen, origin_node, dest_node, weight='travel_time')


# millenium_geom = nodes_proj.loc[origin_node].geometry
# rest_geom = nodes_proj.loc[dest_node].geometry

## San Antonio OSMnx Street Network

In [35]:
ox.settings.useful_tags_way  = ['highway', 'oneway', 'length', 'maxspeed', 'name']
ox.settings.useful_tags_node = ['ref', 'highway']
ox.settings.useful_tags_relation = []

In [ ]:
G_drive_san_antonio = ox.graph_from_polygon(san_antonio_union_poly, network_type='drive', simplify=True)

2026-03-14 03:55:06 Projected GeoDataFrame to +proj=utm +zone=14 +ellps=WGS84 +datum=WGS84 +units=m +no_defs +type=crs
2026-03-14 03:55:06 Projected GeoDataFrame to epsg:4326
2026-03-14 03:55:06 Projected GeoDataFrame to +proj=utm +zone=14 +ellps=WGS84 +datum=WGS84 +units=m +no_defs +type=crs
2026-03-14 03:55:06 Projected GeoDataFrame to epsg:4326
2026-03-14 03:55:06 Requesting data within polygon from API in 71 request(s)
2026-03-14 03:55:06 Retrieved response from cache file "cache/1e78d3d8465ea9d410a53e2c4ff0ce234cf180ee.json"
2026-03-14 03:55:06 Retrieved response from cache file "cache/c1a18b492c3b28e2c8144fb4814e0279562c5a92.json"
2026-03-14 03:55:07 Retrieved response from cache file "cache/9030caf84b862776538fb6eb6a57e2e646598db2.json"
2026-03-14 03:55:07 Retrieved response from cache file "cache/8e651f35476df3b0f14c7f61fd557bf8d958264d.json"
2026-03-14 03:55:07 Retrieved response from cache file "cache/3b416fa581cf3a91b23ea7d852684b4971a139c1.json"
2026-03-14 03:55:07 Retrieve

In [ ]:
ox.save_graphml(G_drive_san_antonio, filepath="data/G_drive_san_antonio.graphml")

In [ ]:
G_drive_san_antonio_proj = ox.project_graph(G_drive_san_antonio, to_crs = "EPSG:3857")
G_drive_san_antonio_proj = ox.add_edge_speeds(G_drive_san_antonio_proj)
G_drive_san_antonio_proj = ox.add_edge_travel_times(G_drive_san_antonio_proj)
ox.save_graphml(G_drive_elpaso, filepath="data/G_drive_san_antonio_proj.graphml")

## Houston OSMnx Street Network

In [ ]:
G_drive_houston = ox.graph_from_polygon(houston_union_poly, bnetwork_type='drive', simplify=True, truncate_by_edge=True)

In [ ]:
G_drive_houston_proj = ox.project_graph(G_drive_houston, to_crs = "EPSG:3857")
G_drive_houston_proj = ox.add_edge_speeds(G_drive_houston_proj)
G_drive_houston_proj = ox.add_edge_travel_times(G_drive_houston_proj)
ox.save_graphml(G_drive_elpaso, filepath="data/G_drive_houston_proj.graphml")

## Dallas OSMnx Street Network

In [ ]:
G_drive_dallas = ox.graph_from_polygon(dallas_union_poly, network_type='drive', simplify=True, truncate_by_edge=True)

In [ ]:
G_drive_dallas_proj = ox.project_graph(G_drive_dallas, to_crs = "EPSG:3857")
G_drive_dallas_proj = ox.add_edge_speeds(G_drive_dallas_proj)
G_drive_dallas_proj = ox.add_edge_travel_times(G_drive_dallas_proj)
ox.save_graphml(G_drive_elpaso, filepath="data/G_drive_dallas_proj.graphml")

# Analysis & Visualization

## Compute drive time isochrone polygons for a list of origin nodes

In [ ]:
def make_isochrone_polys(G, node_ids, travel_times_sec, buffer_m=50):
    """
    Parameters
    ----------
    G              : projected OSMnx DiGraph (must have 'travel_time' edge attribute)
    node_ids       : iterable of OSM node IDs (origin points)
    travel_times_sec : list of cutoffs in seconds, e.g. [300, 600, 1200]
    buffer_m       : buffer to apply to each node point before union (metres)

    Returns
    -------
    GeoDataFrame with columns [travel_time_sec, geometry] in the graph CRS.
    Each row is the *union* of all reachable area across all origin nodes
    at that travel-time cutoff.
    """
    nodes_gdf, _ = ox.graph_to_gdfs(G)
    records = []

    for cutoff in sorted(travel_times_sec, reverse=True):   # largest first → visual layering
        polys = []
        for nid in node_ids:
            if nid not in G.nodes:
                continue
            # ego_graph returns all nodes within `cutoff` seconds of `nid`
            subgraph = nx.ego_graph(G, nid, radius=cutoff, distance='travel_time')
            sub_nodes = nodes_gdf.loc[list(subgraph.nodes())]
            # Buffer each node and union → convex hull gives a filled polygon
            buffered = sub_nodes.geometry.buffer(buffer_m)
            if len(buffered) == 0:
                continue
            poly = unary_union(buffered).convex_hull
            polys.append(poly)

        if polys:
            merged = unary_union(polys)
            records.append({'travel_time_sec': cutoff,
                            'travel_time_min': cutoff // 60,
                            'geometry': merged})

    iso_gdf = gpd.GeoDataFrame(records, crs=nodes_gdf.crs)
    return iso_gdf

In [ ]:
drive_time_thresholds = [300, 600, 1200]
iso_harlingen = make_isochrone_polys(G_drive_harlingen_proj, tower_nodes_harlingen, drive_time_thresholds, buffer_m=100)

# reproject for area calculation
iso_harlingen.to_crs(epsg=4326, inplace=True)   
iso_harlingen["area_km2"] = iso_harlingen.to_crs(epsg=6579).geometry.area / 1e6
print(iso_harlingen[["travel_time_min", "area_km2"]].to_string(index=False))

In [ ]:
iso_eptx = make_isochrone_polys(G_drive_elpaso_proj, tower_nodes_eptx, drive_time_thresholds, buffer_m=100)

# reproject for area calculation
iso_eptx.to_crs(epsg=4326, inplace=True)
iso_eptx["area_km2"] = iso_eptx.to_crs(epsg=6579).geometry.area / 1e6
print(iso_eptx[["travel_time_min", "area_km2"]].to_string(index=False))

## Visualize AOR Isochrone Polygons over the OSM street network with Tower points and Court markers

In [ ]:
def plot_isochrone_map(G_proj, iso_gdf, tower_df, court_df,
                       title="Drive-Time Isochrones around Surveillance Towers",
                       figsize=(14, 12)):
    """
    G_proj    : projected OSMnx graph (EPSG:3857)
    iso_gdf   : isochrone GeoDataFrame (any CRS — will be reprojected)
    tower_df  : tower nearest-node GeoDataFrame (EPSG:3857)
    court_df  : court nearest-node GeoDataFrame (EPSG:3857)
    """
    
    # Isochrone color ramp cutoffs: darkest is 5 min (300 s), 10 min (600 s), lightest is 20 min (1200 s)
    iso_colors = {1200: '#fee8c8', 600: '#fdbb84', 300: '#e34a33'}
    iso_alpha  = {1200: 0.35, 600: 0.50, 300: 0.70}

    fig, ax = plt.subplots(figsize=figsize)

    # Plot road network in light blue-grey
    ox.plot_graph(
        G_proj, ax=ax, bgcolor='white',
        node_size=0, edge_linewidth=0.4, edge_color='#aec7e8',
        show=False, close=False
    )

    # Plot isochrones — largest first so smaller ones layer on top
    iso_plot = iso_gdf.to_crs(epsg=3857)
    
    for _, row in iso_plot.sort_values('travel_time_sec', ascending=False).iterrows():
        cutoff = row['travel_time_sec']
        gpd.GeoDataFrame({'geometry': [row.geometry]}, crs='EPSG:3857').plot(ax=ax, color=iso_colors.get(cutoff, '#999999'), 
                                                                             alpha=iso_alpha.get(cutoff, 0.3), edgecolor='none')

    # Plot towers
    tower_df.plot(ax=ax, color='darkgreen', markersize=15, zorder=4, label='Surveillance Tower')

    # Plot courts
    court_df.plot(ax=ax, color='navy', marker='*', markersize=120, zorder=5, label='Immigration Court')

    # Contextily basemap (OpenStreetMap)
    cx.add_basemap(ax, crs="EPSG:3857", source=cx.providers.CartoDB.Positron, zoom='auto')

    # Legend
    legend_patches = [mpatches.Patch(color='#fee8c8', alpha=0.7, label='20-min drive zone'),
                      mpatches.Patch(color='#fdbb84', alpha=0.7, label='10-min drive zone'),
                      mpatches.Patch(color='#e34a33', alpha=0.8, label=' 5-min drive zone'),
                      mpatches.Patch(color='darkgreen', label='Surveillance Tower'),
                      mpatches.Patch(color='navy', label='Immigration Court'),]
    
    ax.legend(handles=legend_patches, loc='lower left', fontsize=9,
              framealpha=0.9, title='Legend')

    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_axis_off()
    plt.tight_layout()
    return fig, ax

In [ ]:
fig_h, ax_h = plot_isochrone_map(G_drive_harlingen_proj, 
                                 iso_harlingen, 
                                 tower_nodes_harlingen, court_nodes_harlingen,
                                 title="Harlingen AOR — Drive-Time Isochrones Around RVSS/AST Tower Network")
plt.savefig("/home/jovyan/work/GGIS 407/Final Project/figures/harlingen_isochrones.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig_ep, ax_ep = plot_isochrone_map(G_drive_elpaso_proj, 
                                   iso_eptx,
                                   tower_nodes_eptx, court_nodes_eptx, 
                                   title="El Paso AOR — Drive-Time Isochrones Around RVSS/AST Tower Network")
plt.savefig("/home/jovyan/work/GGIS 407/Final Project/figures/elpaso_isochrones.png", dpi=150, bbox_inches='tight')
plt.show()